In [445]:
import pandas as pd
import numpy as np

DATA_FOLDER = "./"
df = pd.read_pickle(DATA_FOLDER + "df_fe_for_ensamble_best_customers_0c.pickle")
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
product_ids = pd.read_csv(DATA_FOLDER + "product_id_apredecir201912.txt", sep="\t")[
    "product_id"
].tolist()
#df = df[df["product_id"].isin(product_ids)]
df = df.sort_values(by=["date_id", "product_id"])
df["customer_id"] = 0
df["target"] = df.groupby(["product_id", "customer_id"])["tn"].shift(-2)


In [446]:
df["mes"][df["mes"] == 8]

5643     8
5644     8
5645     8
5646     8
5647     8
        ..
27759    8
27760    8
27761    8
27762    8
27763    8
Name: mes, Length: 2646, dtype: int8

In [447]:
df.describe()

,product_id,cust_request_qty,cust_request_tn,tn,stock_final,sku_size,year,mes,quarter,date_id,...,prod_tn_lag_1_x_tn_lag_11,prod_tn_lag_1_x_tn_lag_8,prod_tn_lag_3_x_tn_lag_2,prod_tn_lag_3_x_tn_lag_11,prod_tn_lag_3_x_tn_lag_8,prod_tn_lag_2_x_tn_lag_11,prod_tn_lag_2_x_tn_lag_8,prod_tn_lag_11_x_tn_lag_8,customer_id,target
count,31522.000000,31522.000000,31522.000000,31522.000000,13691.000000,31229.000000,31522.000000,31522.000000,31522.000000,31522.000000,...,1.920900e+04,2.224000e+04,2.787000e+04,1.920900e+04,2.224000e+04,1.920900e+04,2.224000e+04,1.920900e+04,31522.0,29076.000000
mean,20535.827073,200.806865,42.924751,42.033772,19.478148,476.827881,2018.037688,6.575471,2.524840,18.027727,...,1.405697e+04,1.394960e+04,1.369290e+04,1.435157e+04,1.411351e+04,1.426182e+04,1.402111e+04,1.518359e+04,0.0,42.737881
std,347.109552,124.339898,113.127739,109.374512,55.627438,883.449097,0.816015,3.452354,1.118599,10.355680,...,1.006608e+05,1.006127e+05,1.004995e+05,9.949145e+04,9.961061e+04,1.020440e+05,9.971995e+04,1.012742e+05,0.0,111.549156
min,20001.000000,0.000000,0.000000,0.000000,-27.311359,1.000000,2017.000000,1.000000,1.000000,0.000000,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000
25%,20239.000000,106.000000,2.221153,2.211540,1.160960,90.000000,2017.000000,4.000000,2.000000,9.000000,...,6.883162e+00,6.546921e+00,6.283891e+00,7.520869e+00,7.176875e+00,7.182292e+00,6.957675e+00,8.685013e+00,0.0,2.222740
50%,20495.000000,185.000000,9.652330,9.598450,5.419600,250.000000,2018.000000,7.000000,3.000000,18.000000,...,1.160452e+02,1.088569e+02,1.063643e+02,1.199704e+02,1.132123e+02,1.201063e+02,1.119023e+02,1.358740e+02,0.0,9.635365
75%,20812.000000,281.000000,29.836285,29.572310,17.584541,475.000000,2019.000000,10.000000,4.000000,27.000000,...,1.034533e+03,9.596051e+02,9.119448e+02,1.041381e+03,9.797971e+02,1.059134e+03,9.570754e+02,1.129115e+03,0.0,29.966728
max,21299.000000,756.000000,2423.708740,2295.198242,1562.024536,10000.000000,2019.000000,12.000000,4.000000,35.000000,...,3.009615e+06,4.261805e+06,4.161229e+06,3.366470e+06,3.375448e+06,3.853622e+06,3.781657e+06,3.374882e+06,0.0,2295.198242


In [448]:
df[["fecha", "date_id"]]

,fecha,date_id
0,2017-01,0
1,2017-01,0
2,2017-01,0
3,2017-01,0
4,2017-01,0
...,...,...
31517,2019-12,35
31518,2019-12,35
31519,2019-12,35
31520,2019-12,35


In [449]:
# transformacion comun de datos para todos los modelos:
# remuevo periodo_min_producto, periodo_max_producto, periodo_min_customer, periodo_max_customer
df = df.drop(columns=["periodo_min_producto", "periodo_max_producto",
                   "periodo_min_customer", "periodo_max_customer"], errors='ignore')

# transformo columnas object a categorical
for col in df.select_dtypes(include=["object"]).columns:
    df[col] = df[col].astype("category")

# transformo plan precios cuidados a categorical
#df["plan_precios_cuidados"] = df["plan_precios_cuidados"].astype("category")

In [450]:

# dropeo columns donde tenga mas sea todo nan hasta date_id 28
print(f"Df shape before dropping columns: {df.shape}")
subset = df[df["date_id"] <= 28]
cols_to_drop = subset.columns[subset.isna().all()]
df = df.drop(columns=cols_to_drop, errors='ignore')
print(f"Df shape after dropping columns: {df.shape}")

Df shape before dropping columns: (31522, 312)
Df shape after dropping columns: (31522, 312)


In [451]:

from sklearn.model_selection import BaseCrossValidator
import numpy as np

class CustomTimeSeriesSplit(BaseCrossValidator):
    def __init__(self, n_splits=3, gap=1):
        self.n_splits = n_splits
        self.gap = gap

    def get_n_splits(self, X=None, y=None, groups=None):
        return self.n_splits

    def split(self, X, y=None, groups=None):
        # Asegurar que X es DataFrame
        
        unique_dates = sorted(X["date_id"].unique(), reverse=True)
        
        for i in range(self.n_splits):
                
            test_date_id = unique_dates[i]
            train_date_id = test_date_id - self.gap - 1
            
            # Usar np.where para obtener posiciones enteras
            train_mask = X["date_id"] <= train_date_id
            test_mask = X["date_id"] == test_date_id
            
            train_idx = np.where(train_mask)[0]
            test_idx = np.where(test_mask)[0]
            
            yield train_idx, test_idx

In [452]:

class LinearRegressionModel:
    def __init__(self, product_ids="magicos", features_to_use=None, especialidad="all"):
        self.model = None
        self.product_ids = product_ids if product_ids is not None else []
        self.features_to_use = features_to_use or ["lags"]
        self.features = []
        self.especialidad = especialidad

    @property
    def name(self):
        return f"LinearRegression-{self.product_ids}-{self.features_to_use}-{self.especialidad}"

    def _get_product_ids(self, df):
        if self.product_ids == "magicos":
            return [20002, 20003, 20006, 20010, 20011, 20018, 20019, 20021,
                    20026, 20028, 20035, 20039, 20042, 20044, 20045, 20046,
                    20049, 20051, 20052, 20053, 20055, 20008, 20001, 20017,
                    20086, 20180, 20193, 20320, 20532, 20612, 20637, 20807,
                    20838]
        elif self.product_ids == "HC":
            hc_products = df[df["cat1"] == "HC"]["product_id"].unique().tolist()
            return hc_products
        elif self.product_ids == "FOODS":
            foods_products = df[df["cat1"] == "FOODS"]["product_id"].unique().tolist()
            return foods_products
        elif self.product_ids == "PC":
            pc_products = df[df["cat1"] == "PC"]["product_id"].unique().tolist()
            return pc_products
        elif self.product_ids == "all":
            return df["product_id"].unique().tolist()
    
    def prepare_dataset(self, df):
        df = df.copy()
        # saco del dataset las rows donde mes==8 para no calcular lags sobre esas rows
        #df = df[df["mes"] != 8]
        df = df.groupby(["product_id", "date_id"], as_index=False).agg({"tn": "sum", "target": "sum", "cat1": "first"}).sort_values(["product_id", "date_id"])
        self.features.append("tn")
        # SACO DEL DATASET LAS ROWS
        if "lags" in self.features_to_use:
            for lag in range(1, 12):
                df[f"tn_{lag}"] = df.groupby("product_id")["tn"].shift(lag)
                self.features.append(f"tn_{lag}")
        if "mean" in self.features_to_use: 
            for window in range(1, 12):
                df[f"tn_mean_{window}"] = df.groupby("product_id")["tn"].transform(lambda x: x.rolling(window=window, min_periods=1).mean())
                self.features.append(f"tn_mean_{window}")
        if "std" in self.features_to_use:
            for window in range(1, 12):
                df[f"tn_std_{window}"] = df.groupby("product_id")["tn"].transform(lambda x: x.rolling(window=window, min_periods=1).std())
                self.features.append(f"tn_std_{window}")
    
        
        return df
    
    def fit_and_predict(self, train_df, pred_df, *args, **kwargs):
        from sklearn.linear_model import LinearRegression
        # si pred_df esta vacio es que es justo el mes 8 asi que lo ignoro
        if pred_df.empty:
            print("Prediccion vacia, no se entrena el modelo.")
            return pd.DataFrame(columns=["product_id", "date_id", "target", "prediction"])
        # si quiero  201912, entreno con 201812 (por estacionalidad)
        # lo busco dinamicamente con pred_df
        date_id_pred = pred_df["date_id"].unique()[0]
        train_df = train_df[train_df["date_id"] == (date_id_pred-12)]
        product_ids_magicos = self._get_product_ids(train_df)

        train_df = train_df[train_df["product_id"].isin(product_ids_magicos)]
        if self.especialidad != "all":
            train_df = train_df[train_df["cat1"] == self.especialidad]

        # elimino registros incompletos
        target = "target"
        train_df = train_df.dropna(subset=self.features + [target])
        print(f"Registros de entrenamiento: {len(train_df)}")
        X = train_df[self.features]
        y = train_df[target]
        model = LinearRegression()
        model.fit(X, y)
        self.model = model

        # hago la prediccion
        pred_df = pred_df.copy()
        
        # solo hace la prediccion para los productos que tienen todas las features
        pred_df = pred_df.dropna(subset=self.features)
        if self.especialidad != "all":
            pred_df = pred_df[pred_df["cat1"] == self.especialidad]
        X_pred = pred_df[self.features]
        pred_df["prediction"] = model.predict(X_pred).clip(min=0)  # Aseguro que la prediccion no sea negativa
        return pd.DataFrame({
            "product_id": pred_df["product_id"],
            "date_id": pred_df["date_id"],
            "target": pred_df["target"],
            "prediction": pred_df["prediction"]
        })
    

In [453]:

class LinearRegressionCombinationModel:
    def __init__(self, product_ids="magicos", features_to_use=None, especialidad="all"):
        self.model = None
        self.product_ids = product_ids if product_ids is not None else []
        self.features_to_use = features_to_use or ["lags"]
        self.features = []
        self.especialidad = especialidad

    @property
    def name(self):
        return f"LinearRegressionCombination-{self.product_ids}-{self.features_to_use}-{self.especialidad}"

    def _get_product_ids(self, df):
        return [20002, 20003, 20006, 20010, 20011, 20018, 20019, 20021,
                    20026, 20028, 20035, 20039, 20042, 20044, 20045, 20046,
                    20049, 20051, 20052, 20053, 20055, 20008, 20001, 20017,
                    20086, 20180, 20193, 20320, 20532, 20612, 20637, 20807,
                    20838]

    
    def prepare_dataset(self, df):
        df = df.copy()
        df = df.groupby(["product_id", "date_id"], as_index=False).agg({"tn": "sum", "target": "sum", "cat1": "first"}).sort_values(["product_id", "date_id"])
        self.features.append("tn")
        if "lags" in self.features_to_use:
            for lag in range(1, 12):
                df[f"tn_{lag}"] = df.groupby("product_id")["tn"].shift(lag)
                self.features.append(f"tn_{lag}")
        if "mean" in self.features_to_use: 
            for window in range(1, 12):
                df[f"tn_mean_{window}"] = df.groupby("product_id")["tn"].transform(lambda x: x.rolling(window=window, min_periods=1).mean())
                self.features.append(f"tn_mean_{window}")
        if "std" in self.features_to_use:
            for window in range(1, 12):
                df[f"tn_std_{window}"] = df.groupby("product_id")["tn"].transform(lambda x: x.rolling(window=window, min_periods=1).std())
                self.features.append(f"tn_std_{window}")
    
        
        return df
    
    def fit_and_predict(self, train_df, pred_df, *args, **kwargs):
        from sklearn.linear_model import LinearRegression
        # si quiero  201912, entreno con 201812 (por estacionalidad)
        # lo busco dinamicamente con pred_df
        date_id_pred = pred_df["date_id"].unique()[0]
        train_df = train_df[train_df["date_id"] == (date_id_pred-12)]
        product_ids_magicos = self._get_product_ids(train_df)
        target = "target"

        train_df = train_df[train_df["product_id"].isin(product_ids_magicos)]
        if self.especialidad != "all":
            train_df_especialidad = train_df[train_df["cat1"] == self.especialidad].dropna(subset=self.features + [target]).copy()
            print(f"product ids especialidad: {train_df_especialidad['product_id'].unique().tolist()}")

        # elimino registros incompletos
        train_df = train_df.dropna(subset=self.features + [target])
        print(f"Registros de entrenamiento: {len(train_df)}")
        X = train_df[self.features]
        y = train_df[target]
        model = LinearRegression()
        model.fit(X, y)
        self.model = model

        if self.especialidad != "all":
            X_especialidad = train_df_especialidad[self.features]
            y_especialidad = train_df_especialidad[target]
            model_especialidad = LinearRegression()
            model_especialidad.fit(X_especialidad, y_especialidad)
            self.model_especialidad = model_especialidad

        # hago la prediccion
        pred_df = pred_df.copy()
        
        # solo hace la prediccion para los productos que tienen todas las features
        pred_df = pred_df.dropna(subset=self.features)
        if self.especialidad != "all":
            pred_df_especialidad = pred_df[pred_df["cat1"] == self.especialidad]
        X_pred = pred_df[self.features]
        pred_df["prediction"] = model.predict(X_pred).clip(min=0)  # Aseguro que la prediccion no sea negativa
        if self.especialidad != "all":
            X_pred_especialidad = pred_df_especialidad[self.features]
            pred_df_especialidad["prediction"] = model_especialidad.predict(X_pred_especialidad).clip(min=0)
            # reemplazo las predicciones de pred_df en los product_id de pred_df_especialidad
            pred_df.loc[pred_df["cat1"] == self.especialidad, "prediction"] = pred_df_especialidad["prediction"]
        return pd.DataFrame({
            "product_id": pred_df["product_id"],
            "date_id": pred_df["date_id"],
            "target": pred_df["target"],
            "prediction": pred_df["prediction"]
        })
    

In [454]:
class LinearRegressionByProductModel:
    def __init__(self,):
        self.model = None
        self.features = []
    
    @property
    def name(self):
        return "LinearRegressionByProduct"

    
    def prepare_dataset(self, df):
        df = df.copy()
        df = df.groupby(["product_id", "date_id"], as_index=False).agg({"tn": "sum", "target": "sum", "cat1": "first"}).sort_values(["product_id", "date_id"])
        self.features.append("tn")
        for lag in range(1, 24):
            df[f"tn_{lag}"] = df.groupby("product_id")["tn"].shift(lag)
            self.features.append(f"tn_{lag}") 
        return df
    
    def fit_and_predict(self, train_df, pred_df, *args, **kwargs):
        from sklearn.linear_model import LinearRegression
        # si quiero  201912, entreno con 201812 (por estacionalidad)
        # lo busco dinamicamente con pred_df
        date_id_pred = pred_df["date_id"].unique()[0]
        train_df = train_df[train_df["date_id"] == (date_id_pred-12)]

        models = {}
        for product_id in product_ids:
            product_id_train_df = train_df[train_df["product_id"] == product_id]
            target = "target"
            product_id_train_df = product_id_train_df.dropna(subset=[target])
            # hago fillna de features con el promedio del resto de las features en esa row en particular NO CON LA MEDIA DE TODO EL DATASET
            product_id_train_df[self.features] = product_id_train_df[self.features].fillna(
                product_id_train_df[self.features].mean(axis=0)
            ).fillna(0)
            if len(product_id_train_df) == 0:
                continue
            X = train_df[self.features]
            y = train_df[target]
            model = LinearRegression()
            model.fit(X, y)
            self.model = model
            models[product_id] = model        # hago la prediccion
        pred_df = pred_df.copy()
        
        pred_df["prediction"] = np.nan  # Inicializo la columna de predicciones
        for product_id in product_ids:
            model = models.get(product_id)
            if model is None:
                continue
            
            product_id_pred_df = pred_df[pred_df["product_id"] == product_id]
            
            # solo hace la prediccion para los productos que tienen todas las features
            #hago el fillna de features con el promedio del resto de las features en esa row en particular NO CON LA MEDIA DE TODO EL DATASET
            product_id_pred_df[self.features] = product_id_pred_df[self.features].fillna(
                product_id_pred_df[self.features].mean(axis=0)
            )
            X_pred = product_id_pred_df[self.features]
            y_pred = model.predict(X_pred).clip(min=0)
            pred_df.loc[pred_df["product_id"] == product_id, "prediction"] = y_pred
        
        # drop nan predictions
        pred_df = pred_df.dropna(subset=["prediction"])
        return pd.DataFrame({
            "product_id": pred_df["product_id"],
            "date_id": pred_df["date_id"],
            "target": pred_df["target"],
            "prediction": pred_df["prediction"]
        })
    

In [455]:
class SimpleMovingAveragePredictor:
    ''' Usa una media movil simple para predecir tn'''
    def __init__(self, window_size=12):
        self.model = None
        self.window_size = window_size

    @property
    def name(self):
        return f"SMA-{self.window_size}"
    
    def prepare_dataset(self, df):
        df = df.copy()
        df = df.groupby(["product_id", "date_id"], as_index=False).agg({"tn": "sum", "target": "sum"}).sort_values(["product_id", "date_id"])
        # hago una columna que es la media movil simple agrupada por producto
        df["tn_sma"] = df.groupby("product_id")["tn"].transform(
            lambda x: x.rolling(window=self.window_size, min_periods=1).mean()
        )
        return df
    
    def fit_and_predict(self, train_df, pred_df, *args, **kwargs):

        return pd.DataFrame({
            "product_id": pred_df["product_id"],
            "date_id": pred_df["date_id"],
            "target": pred_df["target"],
            "prediction": pred_df["tn_sma"],
        })

In [456]:
class ExponentialMovingAveragePredictor:
    ''' Usa una media movil exponencial para predecir tn'''
    def __init__(self, window_size=12):
        self.model = None
        self.window_size = window_size

    @property
    def name(self):
        return f"EMA-{self.window_size}"
    def prepare_dataset(self, df):
        df = df.copy()
        df = df.groupby(["product_id", "date_id"], as_index=False).agg({"tn": "sum", "target": "sum"}).sort_values(["product_id", "date_id"])
        # hago una columna que es la media movil exponencial agrupada por producto
        df["tn_ema"] = df.groupby("product_id")["tn"].transform(
            lambda x: x.ewm(span=self.window_size, adjust=False).mean()
        )
        return df
    
    def fit_and_predict(self, train_df, pred_df, *args, **kwargs):
        return pd.DataFrame({
            "product_id": pred_df["product_id"],
            "date_id": pred_df["date_id"],
            "target": pred_df["target"],
            "prediction": pred_df["tn_ema"],
        })

In [457]:
import numpy as np
from scipy.optimize import minimize

class MediaMovilPonderada:
    def __init__(self, window_size=12):
        self.window_size = window_size
        self.weights = None

    @property
    def name(self):
        return f"WMA-{self.window_size}"
    
    def prepare_dataset(self, df):
        return df.groupby(["product_id", "date_id"], as_index=False).agg({"tn": "sum", "target": "sum"}).sort_values(["product_id", "date_id"])
    
    def fit_and_predict(self, train_df, pred_df, *args, **kwargs):        
        # Optimizar pesos con últimas 3 fechas
        last_dates = sorted(train_df["date_id"].unique())[-3:]
        opt_data = train_df[train_df["date_id"].isin(last_dates)]
        
        def loss(weights):
            total_error = 0
            for _, group in opt_data.groupby("product_id"):
                for i, row in group.iterrows():
                    hist = train_df[(train_df["product_id"] == row["product_id"]) & 
                                   (train_df["date_id"] < row["date_id"])]["tn"].values[-self.window_size:]
                    if len(hist) > 0:
                        pred = np.dot(hist, weights[-len(hist):])
                        total_error += (pred - row["target"]) ** 2
            return total_error
        
        self.weights = minimize(loss, np.ones(self.window_size), method='L-BFGS-B').x
        
        # Aplicar predicción
        all_df = pd.concat([train_df, pred_df])
        all_df["wma"] = np.nan
        
        for pid, group in all_df.groupby("product_id"):
            for i, (idx, row) in enumerate(group.iterrows()):
                hist = group["tn"].iloc[:i].values[-self.window_size:]
                if len(hist) > 0:
                    all_df.loc[idx, "wma"] = np.dot(hist, self.weights[-len(hist):])
        
        pred_out = all_df.loc[pred_df.index]
        return pd.DataFrame({
            "product_id": pred_out["product_id"],
            "date_id": pred_out["date_id"], 
            "target": pred_out["target"],
            "prediction": pred_out["wma"]
        })    

In [458]:
class AutoGluonPredictor:
    def __init__(self, presets="best_quality", estimator=None):
        self.model = None
        self.presets = presets
        self.estimator = estimator

    @property
    def name(self):
        if self.estimator:
            return f"AutoGluon-{self.estimator}"
        return f"AutoGluon-{self.presets}"
    
    def prepare_dataset(self, df):

        df = df.copy()
        df["fecha"] = df["fecha"].apply(lambda x: x.to_timestamp("M"))
        df = df.rename(columns={"fecha": "timestamp"})
        df["product_id"] = df["product_id"].astype(int)
        df["serie_id"] = df["product_id"].astype(str) + "-" + df["customer_id"].astype(str)
        df["cat1"] = df["cat1"].astype("category")
        df["cat2"] = df["cat2"].astype("category")
        df["cat3"] = df["cat3"].astype("category")
        df["brand"] = df["brand"].astype("category")
        df["sku_size"] = df["sku_size"].astype("category")

        self.static_features_df = pd.DataFrame({
            "cat1": df.groupby("serie_id")["cat1"].first(),
            "cat2": df.groupby("serie_id")["cat2"].first(),
            "cat3": df.groupby("serie_id")["cat3"].first(),
            "brand": df.groupby("serie_id")["brand"].first(),
            "sku_size": df.groupby("serie_id")["sku_size"].first(),
            "customer_id": df.groupby("serie_id")["customer_id"].first(),
            "product_id": df.groupby("serie_id")["product_id"].first(),
        }).reset_index()
        

        min_periods = 12  # Mínimo 6 meses de datos
        product_counts = df.groupby(["product_id"]).size()
        valid_products = product_counts[product_counts >= min_periods].index
        df = df[df["product_id"].isin(valid_products)]        
        df = df.dropna(subset=["tn"])
        return df
    
    def fit_and_predict(self, train_df, pred_df, df_model):
        from autogluon.timeseries import TimeSeriesPredictor, TimeSeriesDataFrame
        # el autogluon lo entreno con todas las fechas hasta pred_df
        train_df = df_model[df_model["date_id"] <= pred_df["date_id"].unique()[0]]
        train_df = train_df[train_df["product_id"].isin(product_ids)]
        ts_data = TimeSeriesDataFrame.from_data_frame(
            train_df.drop(columns=["target"]), 
            id_column="serie_id", 
            timestamp_column="timestamp", 
            static_features_df=self.static_features_df
        )
        ts_data = ts_data.sort_index()
        ts_data = ts_data.fill_missing_values()

        predictor = TimeSeriesPredictor(
            prediction_length=2,
            target="tn",
            freq="MS",
        )
        if self.estimator:
            predictor.fit(ts_data, hyperparameters={self.estimator: {}})
        else:
            predictor.fit(ts_data, presets=self.presets)
        forecast = predictor.predict(ts_data)
        forecast_mean = forecast["mean"].reset_index()
        forecast_mean = forecast_mean[forecast_mean["timestamp"] == forecast_mean["timestamp"].max()]
        forecast_mean[["product_id", "customer_id"]] = forecast_mean["item_id"].str.split("-", expand=True)
        forecast_mean["product_id"] = forecast_mean["product_id"].astype(int)
        forecast_mean = forecast_mean.groupby("product_id").agg({
            "mean": "sum",
        }).reset_index()

        # rename item_id to product_id
        pred_df = pred_df.copy()
        pred_df["product_id"] = pred_df["product_id"].astype(int)
        # separo serie_id en product_id y customer_id
        pred_df = pred_df.groupby(["product_id"]).agg({
            "target": "sum",
            "date_id": "first"
        }).reset_index()
        pred_df = pred_df.merge(forecast_mean, on=["product_id"], how="left")
        pred_df = pred_df.rename(columns={"mean": "prediction"})
        return pd.DataFrame({
            "product_id": pred_df["product_id"],
            "date_id": pred_df["date_id"],
            "target": pred_df["target"],
            "prediction": pred_df["prediction"]
        })



In [459]:


class BaseTabularPredictor:
    
    def _scaling_df(self, df, train=True):
        df = df.copy()
        import re
        numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        transformations = {
            "tn": [
                r"tn$",
                r"cust_request_qty_per_tn$",
                r"tn_lag_*",
                r"tn_rolling_mean_*",
                r"tn_rolling_max_*",
                r"tn_rolling_min_*",
                r"tn_.*_vendidas$",
                r"tn_agg*",
                r"tn_wavelet_*",
            ]
            + [r"stock_final$"]
            + [r"cust_request_tn_minus_tn$"]
            + [r"tn_diff_*"],
            "cust_request_qty": [
                r"cust_request_qty$",
                r"cust_request_qty_lag_*",
                r"cust_request_qty_rolling_mean_*",
                r"cust_request_qty_rolling_max_*",
                r"cust_request_qty_rolling_min_*",
                r"cust_request_qty_.*_vendidas$",
                r"cust_request_qty_agg*",
                r"cust_request_qty_wavelet_*",
            ]
            + [r"cust_request_qty_diff_*"],
        }

        # busco todas las columnas que empiezan con prod_ y agrego key y valor en transformation
        for col in numeric_cols:
            if col.startswith("prod_"):
                transformations[col] = [r"{}$".format(col)]

        from pandas.errors import PerformanceWarning
        import warnings
        warnings.simplefilter(action="ignore", category=PerformanceWarning)
        df = df.set_index(['serie_id', "date_id"])
        if train:
            prod_stats = df.groupby(["serie_id"])[
                list(transformations.keys())
            ].agg(["std"])
            prod_stats.columns = [
                f"{col[0]}_{col[1]}" for col in prod_stats.columns
            ]  # renombro las columnas para que no tengan tupla

            prod_stats = prod_stats.reset_index()
            self.prod_stats = prod_stats
            prod_stats = prod_stats.set_index(['serie_id'])
            # supress performance warnings
            self.prod_stats = prod_stats

            print("Scaling")
        else:
            if self.prod_stats is None:
                raise ValueError("prod_stats is not set. Call prepare_dataset first.")
            prod_stats = self.prod_stats
        for trainer, regex_cols in transformations.items():
            for col in regex_cols:
                matching_cols = [c for c in numeric_cols if re.match(col, c)]
                if not matching_cols:
                    continue
                for col in matching_cols:
                    std_col = prod_stats[trainer + "_std"]
                    df[f"{col}_scaled"] = (df[col] / std_col).replace([np.inf, -np.inf], np.nan)

        # scalo el target con tn_std
        df["target_scaled"] = df["target"] / prod_stats["tn_std"]
        df["target_scaled"] = df["target_scaled"].replace([np.inf, -np.inf], np.nan).fillna(0)

        df = df.reset_index()
        return df

    def prepare_dataset(self, df):
        df = df.copy()
        df["serie_id"] = df["product_id"].astype(str) + "-" + df["customer_id"].astype(str)
        return df


class AutoMLPredictor(BaseTabularPredictor):
    
    def __init__(self, estimator="lgbm", time_budget=60):
        self.model = None
        self.prod_stats = None
        self.estimator = estimator
        self.time_budget = time_budget

    @property
    def name(self):
        return f"AutoML-{self.estimator}-{self.time_budget}s"
    
    def custom_metric(self, X_val, y_val, estimator, labels, X_train, y_train, *args, **kwargs):
        y_pred = estimator.predict(X_val)
    
        temp_df = pd.DataFrame({
            "product_id": X_val["product_id"].values,
            "customer_id": X_val["customer_id"].values,
            "y_true": y_val,
            "y_pred": y_pred
        })
        temp_df["product_id"] = temp_df["product_id"].astype(int)
        temp_df["customer_id"] = temp_df["customer_id"].astype(int)
        prod_stats = self.prod_stats.copy().reset_index()
        prod_stats[["product_id", "customer_id"]] = prod_stats["serie_id"].str.split("-", expand=True)
        prod_stats["product_id"] = prod_stats["product_id"].astype(int)
        prod_stats["customer_id"] = prod_stats["customer_id"].astype(int)
        temp_df = temp_df.merge(prod_stats[["product_id", "customer_id", "tn_std"]], on=["product_id", "customer_id"], how="left")
        # desescale the predictions
        temp_df["y_pred"] = temp_df["y_pred"] * temp_df["tn_std"]
        temp_df["y_true"] = temp_df["y_true"] * temp_df["tn_std"]
    
        grouped = temp_df.groupby("product_id")[["y_true", "y_pred"]].sum()
        total_true = grouped["y_true"].sum()
    
        if total_true == 0:
            return 0.0, {"total_error": 0.0}
    
        total_error = np.abs(grouped["y_pred"] - grouped["y_true"]).sum() / total_true
        return total_error, {"total_error": total_error}

    def fit_and_predict(self, train_df, pred_df, df_model):
        from flaml import AutoML
        train_df = self._scaling_df(train_df, train=True)
        pred_df = self._scaling_df(pred_df, train=False)
        tscv = CustomTimeSeriesSplit(2, gap=1)
        automl_settings = {
            "time_budget": self.time_budget,
            "task": "regression",
            "metric": self.custom_metric,
            "estimator_list": [self.estimator],
            "n_jobs": -1,
            "eval_method": "cv",
            "split_type": tscv,
            "verbose": 3,
            "retrain_full": True
        }
        X_train = train_df.drop(columns=["target", "target_scaled", "fecha"])
        y_train = train_df["target_scaled"]
        automl = AutoML()
        automl.fit(X_train, y_train, **automl_settings)
        # hago la prediccion
        y_pred = automl.predict(pred_df.drop(columns=["target", "target_scaled", "fecha"]))
        pred_df = pred_df.copy()
        pred_df["prediction"] = y_pred
        pred_df["serie_id"] = pred_df["product_id"].astype(str) + "-" + pred_df["customer_id"].astype(str)
        pred_df.set_index("serie_id", inplace=True)
        pred_df["prediction"] = pred_df["prediction"] * self.prod_stats["tn_std"]
        pred_df = pred_df.reset_index().groupby("product_id").agg({
            "target": "sum",
            "date_id": "first",
            "prediction": "sum"
        }).reset_index()
 
        return pd.DataFrame({
            "product_id": pred_df["product_id"],
            "date_id": pred_df["date_id"],
            "target": pred_df["target"],
            "prediction": pred_df["prediction"]
        })
        


In [460]:
class AutoGluonTabularPredictor(BaseTabularPredictor):
    
    def __init__(self, presets="medium_quality", exclude_model_types=None, time_budget=None, subsample=1):
        self.model = None
        self.prod_stats = None
        self.presets = presets
        self.exclude_model_types = exclude_model_types or []
        self.time_budget = time_budget
        self.subsample = subsample

    @property
    def name(self):
        return f"AutoGluonTabular-{self.presets}-budget-{self.time_budget}s-subsample-{self.subsample}"
    
    def fit_and_predict(self, train_df, pred_df, df_model):
        from autogluon.tabular import TabularPredictor, TabularDataset
        train_df = self._scaling_df(train_df, train=True)
        pred_df = self._scaling_df(pred_df, train=False)

        # uso el date_id mas alto de train_df como tunning_data
        train_df = train_df.sample(frac=self.subsample, random_state=42)
        train = TabularDataset(train_df.drop(columns=["fecha", "serie_id"], errors='ignore'))
        pred = TabularDataset(pred_df.drop(columns=["fecha", "serie_id"], errors='ignore'))

        predictor = TabularPredictor(
            label="target_scaled",
            eval_metric="mean_absolute_error",
        )
        predictor.fit(
            train.drop(columns=["target"]),
            presets=self.presets,
            excluded_model_types=["RF", "XT"] + self.exclude_model_types,
            time_limit=self.time_budget,
        )

        y_pred = predictor.predict(pred.drop(columns=["target", "target_scaled"]))
        pred_df = pred_df.copy()
        pred_df["prediction"] = y_pred
        pred_df["serie_id"] = pred_df["product_id"].astype(str) + "-" + pred_df["customer_id"].astype(str)
        pred_df.set_index("serie_id", inplace=True)
        pred_df["prediction"] = pred_df["prediction"] * self.prod_stats["tn_std"]
        pred_df = pred_df.reset_index().groupby("product_id").agg({
            "target": "sum",
            "date_id": "first",
            "prediction": "sum"
        }).reset_index()
 
        return pd.DataFrame({
            "product_id": pred_df["product_id"],
            "date_id": pred_df["date_id"],
            "target": pred_df["target"],
            "prediction": pred_df["prediction"]
        })

In [461]:
class BasicXGBoostPredictor(BaseTabularPredictor):
    
    def __init__(self):
        self.model = None
        self.prod_stats = None

    @property
    def name(self):
        return f"BasicXGBoost"
    
    def fit_and_predict(self, train_df, pred_df, df_model):
        import xgboost as xgb
        train_df = self._scaling_df(train_df, train=True)
        pred_df = self._scaling_df(pred_df, train=False)

        X_train = train_df.drop(columns=["target", "target_scaled", "fecha", "serie_id"])
        y_train = train_df["target_scaled"]

        X_test = pred_df.drop(columns=["target", "target_scaled", "fecha", "serie_id"])
        y_test = pred_df["target_scaled"]
        
        # I use the tn_std as weight
        train_df = train_df.merge(self.prod_stats.reset_index()[["serie_id", "tn_std"]], on="serie_id", how="left")
        w_train = train_df["tn_std"].fillna(0)
        dtrain = xgb.DMatrix(X_train, label=y_train, weight=w_train, enable_categorical=True)
        dtest = xgb.DMatrix(X_test, label=y_test, enable_categorical=True)
        
        model = xgb.train(
            params={
                "objective": "reg:tweedie",
                "device": "cuda",
                "tree_method": "hist",
                "sampling_method": "uniform",
                "max_depth": 0,
                "learning_rate": 0.03,
                "num_leaves": 31,
                "subsample": 0.8,
                "colsample_bytree": 0.6,
            },
            dtrain=dtrain,
            evals=[(dtest, "test")],
            num_boost_round=1000,
            #num_boost_round=20
        )

        y_pred = model.predict(dtest)
        pred_df = pred_df.copy()
        pred_df["prediction"] = y_pred
        pred_df["serie_id"] = pred_df["product_id"].astype(str) + "-" + pred_df["customer_id"].astype(str)
        pred_df.set_index("serie_id", inplace=True)
        pred_df["prediction"] = pred_df["prediction"] * self.prod_stats["tn_std"]
        pred_df = pred_df.reset_index().groupby("product_id").agg({
            "target": "sum",
            "date_id": "first",
            "prediction": "sum"
        }).reset_index()
 
        return pd.DataFrame({
            "product_id": pred_df["product_id"],
            "date_id": pred_df["date_id"],
            "target": pred_df["target"],
            "prediction": pred_df["prediction"]
        })

In [462]:
class BasicLGBMPredictor(BaseTabularPredictor):
    def __init__(self, with_scaling=True, extra_trees=False, n_trials=0, boosting_type="gbdt", use_weight=True, target="t+2"):
        self.model = None
        self.prod_stats = None
        self.with_scaling = with_scaling
        self.extra_trees = extra_trees
        self.n_trials = n_trials
        if self.n_trials == 0:
            self.n_trials = 1
        self.boosting_type = boosting_type
        self.base_params = {
            "objective": "tweedie",
            "device": "cpu",
            "max_bin": 512,
            "extra_trees": self.extra_trees,
            "boosting_type": self.boosting_type,
            "metric": "None",
        }
        self.use_weight = use_weight
        self.target = target # opciones: t+2, delta
        if self.target == "delta" or self.target == "logdiff":
            self.base_params["objective"] = "regression"


    @property
    def name(self):
        return f"LGBM-extra_trees-{self.extra_trees}-trials-{self.n_trials}-scaling-{self.with_scaling}-boosting-{self.boosting_type}-weight-{self.use_weight}-target-{self.target}"


    def optimize_params(self,train_df):
        class LGBCustomMetric:
            def __init__(self, eval_df, prod_stats):
                self.eval_df = eval_df.copy()
                self.eval_df["serie_id"] = self.eval_df["product_id"].astype(str) + "-" + self.eval_df["customer_id"].astype(str)
                self.prod_stats = prod_stats

            def __call__(self, preds, train_data):
                eval_df = self.eval_df.copy()
                eval_df["prediction"] = preds

                eval_df = eval_df[eval_df["product_id"].isin(product_ids)]
                eval_df.set_index("serie_id", inplace=True)
                eval_df["prediction"] = eval_df["prediction"] * self.prod_stats["tn_std"]
                eval_df = eval_df.groupby("product_id").agg({
                    "target": "sum",
                    "prediction": "sum",
                }).reset_index()
                total_error = np.sum(np.abs(eval_df["prediction"] - eval_df["target"])) / np.sum(eval_df["target"])
                return "total_error", total_error, False  # False indicates that lower is better

        #uso la maxima fecha como eval_df
        eval_df = train_df[train_df["date_id"] == train_df["date_id"].max()].copy()
        train_df = train_df[train_df["date_id"] < eval_df["date_id"].max()].copy()
        from optuna import create_study
        from optuna.samplers import TPESampler
        import lightgbm as lgb
        def objective(trial):
            nonlocal train_df
            nonlocal eval_df
            train_df_trial = train_df.copy()
            eval_df_trial = eval_df.copy()
            if self.n_trials <= 1:
                print("Skipping hyperparameter optimization, using default parameters.")
                params = {
                    **self.base_params,
                    "num_leaves": 31,
                    "learning_rate": 0.03,
                    "feature_fraction": 0.8,
                    "bagging_fraction": 0.8,
                    "bagging_freq": 5,
                    "min_data_in_leaf": 30,
                }
            else:
                print("Optimizing hyperparameters with Optuna.")        
                params = {
                    **self.base_params,
                    "tweedie_variance_power": trial.suggest_float("tweedie_variance_power", 1.1, 1.9),
                    "num_leaves": trial.suggest_int("num_leaves", 16, 512),
                    "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.075),
                    "feature_fraction": trial.suggest_float("feature_fraction", 0.3, 1.0),
                    "bagging_fraction": trial.suggest_float("bagging_fraction", 0.3, 1.0),
                    "bagging_freq": trial.suggest_int("bagging_freq", 1, 20),
                    "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 1, 40),
                }
                if self.target == "delta":
                    # en este caso saco tweedie_variance_power porque no es necesario
                    params.pop("tweedie_variance_power", None)
            # TODO: optimizar custom metric porque relentiza mucho, mientras uso mae
            params["metric"] = "mae"
            # TODO: en la optimizacion NO Uso max bin por performance, pero en el entrenamiento si
            params["max_bin"] = 255
            # agrego los weights a train_df (y luego hago drop de la columna)
            train_df_trial = train_df_trial.merge(self.prod_stats.reset_index()[["serie_id", "tn_std"]], on="serie_id", how="left")
            w_train = train_df_trial["tn_std"].fillna(0)
            train_df_trial.drop(columns=["tn_std"], inplace=True)
            # si use_weight es False pongo w_train a 1.0
            if not self.use_weight:
                w_train = np.ones_like(w_train)

            dtrain = lgb.Dataset(train_df_trial.drop(columns=["target", "target_scaled", "fecha", "serie_id"]), label=train_df_trial["target_scaled"], weight=w_train)
            dval = lgb.Dataset(eval_df_trial.drop(columns=["target", "target_scaled", "fecha", "serie_id"]), label=eval_df_trial["target_scaled"])
            eval_result = {}
            if self.boosting_type == "gbdt":
                callbacks = [
                    lgb.log_evaluation(500),
                    lgb.early_stopping(int(400 + 4 / params["learning_rate"]), first_metric_only=True),
                    lgb.record_evaluation(eval_result),
                ]
            else:
                # dart no tiene early stopping
                callbacks = [
                    lgb.log_evaluation(500),
                    lgb.record_evaluation(eval_result),
                ]
            model = lgb.train(
                params, 
                dtrain, 
                num_boost_round=9999, 
                valid_sets=[dval], 
                valid_names=["eval"],
                #feval=LGBCustomMetric(eval_df, self.prod_stats),
                callbacks=callbacks
            )
            scores = eval_result["eval"]["l1"]
            best_iteration = min(enumerate(scores, 1), key=lambda x: x[1])[0]  # Encuentra la mejor iteración
            print(f"Best iteration: {best_iteration}, Score: {scores[best_iteration-1]}")
            # me guardo la mejor iteracion como attr del trial
            trial.set_user_attr("best_iteration", best_iteration)
            y_pred = model.predict(eval_df_trial.drop(columns=["target","target_scaled", "fecha", "serie_id"]), num_iteration=best_iteration)
            eval_df_trial["prediction"] = y_pred
            eval_df_trial.set_index("serie_id", inplace=True)
            eval_df_trial["prediction"] = eval_df_trial["prediction"] * self.prod_stats["tn_std"]
            if self.target == "delta":
                eval_df_trial["prediction"] = eval_df_trial["prediction"] + eval_df_trial["tn"]
                eval_df_trial["target"] = eval_df_trial["target"] + eval_df_trial["tn"]
            elif self.target == "logdiff":
                # inversa: target = exp(target + np.log(tn + 1)) - 1
                eval_df_trial["prediction"] = np.exp(eval_df_trial["prediction"] + np.log(eval_df_trial["tn"] + 1)) - 1
                eval_df_trial["target"] = np.exp(eval_df_trial["target"] + np.log(eval_df_trial["tn"] + 1)) - 1
            eval_df_trial = eval_df_trial.reset_index().groupby("product_id").agg({
                "target": "sum",
                "prediction": "sum",
            }).reset_index()
            total_error = np.sum(np.abs(eval_df_trial["prediction"] - eval_df_trial["target"])) / np.sum(eval_df_trial["target"])
            return total_error

        study = create_study(direction="minimize", sampler=TPESampler(seed=42))
        if self.n_trials == 0:
            self.n_trials = 1 # hago uno para optimizar al menos el n_estimator
        study.optimize(objective, n_trials=self.n_trials, show_progress_bar=True)
        best_trial = study.best_trial
        best_parameters = best_trial.params
        num_iteration = best_trial.user_attrs.get("best_iteration")
        if self.n_trials <= 1:
            return {
                "num_leaves": 31,
                "learning_rate": 0.03,
                "feature_fraction": 0.8,
                "bagging_fraction": 0.8,
                "bagging_freq": 5,
                "min_data_in_leaf": 30,
            }, num_iteration
        return best_parameters, num_iteration

    def fit_and_predict(self, train_df, pred_df, df_model):
        import lightgbm as lgb
        # si target es delta hago la diff entre target y tn
        if self.target == "delta":
            train_df["target"] = train_df["target"] - train_df["tn"]
            pred_df["target"] = pred_df["target"] - pred_df["tn"]
        elif self.target == "logdiff":
            train_df["target"] = np.log(train_df["target"] + 1) - np.log(train_df["tn"] + 1)
            # inversa: target = exp(target + np.log(tn + 1)) - 1
            pred_df["target"] = np.log(pred_df["target"] + 1) - np.log(pred_df["tn"] + 1)

        if self.with_scaling:
            ## Nota: puedo escalar con los datos de pred porque en el instante T tengo esos datos, lo que no tengo es el target
            #train_scaling_df = df_model[df_model["date_id"] <= pred_df["date_id"].max()]
            #train_scaling_df = self._scaling_df(train_scaling_df, train=True)
            train_df = self._scaling_df(train_df, train=True)
            pred_df = self._scaling_df(pred_df, train=False)
        else:
            # ignore SettingWithCopyWarning
            import warnings
            warnings.filterwarnings("ignore", category=pd.errors.SettingWithCopyWarning)
            self.prod_stats = pd.DataFrame({
                "serie_id": train_df["serie_id"].unique(),
                "tn_std": 1.0
            }).set_index("serie_id")
            train_df["target_scaled"] = train_df["target"]
            pred_df["target_scaled"] = pred_df["target"]

        best_params, num_iteration = self.optimize_params(train_df)
        X_train = train_df.drop(columns=["target", "target_scaled", "fecha", "serie_id"])
        y_train = train_df["target_scaled"]

        X_test = pred_df.drop(columns=["target", "target_scaled", "fecha", "serie_id"])
        y_test = pred_df["target_scaled"]
        
        # I use the tn_std as weight
        train_df = train_df.merge(self.prod_stats.reset_index()[["serie_id", "tn_std"]], on="serie_id", how="left")
        w_train = train_df["tn_std"].fillna(0)
        # si use_weight es False pongo w_train a 1.0
        if not self.use_weight:
            w_train = np.ones_like(w_train)
        
        dtrain = lgb.Dataset(X_train, label=y_train, weight=w_train)
        dtest = lgb.Dataset(X_test, label=y_test)

        params = {
            **self.base_params,
            **best_params,
            "verbose": 0,
        }
        print(f"Training LGBM with parameters: {params}, and {num_iteration} iterations")
        model = lgb.train(
            params,
            dtrain,
            #num_boost_round=1000,
            num_boost_round=num_iteration,
            valid_sets=[dtest],
            callbacks=[lgb.log_evaluation(1000)]
        )

        y_pred = model.predict(X_test)
        pred_df = pred_df.copy()
        pred_df["prediction"] = y_pred
        pred_df["serie_id"] = pred_df["product_id"].astype(str) + "-" + pred_df["customer_id"].astype(str)
        pred_df.set_index("serie_id", inplace=True)
        pred_df["prediction"] = pred_df["prediction"] * self.prod_stats["tn_std"]
        if self.target == "delta":
            # si target es delta, deshago la diff
            pred_df["prediction"] = pred_df["prediction"] + pred_df["tn"]
            pred_df["target"] = pred_df["target"] + pred_df["tn"]
        elif self.target == "logdiff":
            # si target es logdiff, deshago la log diff
            pred_df["prediction"] = np.exp(pred_df["prediction"] + np.log(pred_df["tn"] + 1)) - 1
            pred_df["target"] = np.exp(pred_df["target"] + np.log(pred_df["tn"] + 1)) - 1
        pred_df = pred_df.reset_index().groupby("product_id").agg({
            "target": "sum",
            "date_id": "first",
            "prediction": "sum"
        }).reset_index()

        return pd.DataFrame({
            "product_id": pred_df["product_id"],
            "date_id": pred_df["date_id"],
            "target": pred_df["target"],
            "prediction": pred_df["prediction"]
        })

In [463]:
#autogluon_tabular_predictor = AutoGluonTabularPredictor(presets="medium")
#df_autogluon_tabular = autogluon_tabular_predictor.prepare_dataset(df)
#test_df = df_autogluon_tabular[df_autogluon_tabular["date_id"] == 33]
#train_df = df_autogluon_tabular[df_autogluon_tabular["date_id"] < 32]
#results = autogluon_tabular_predictor.fit_and_predict(train_df, test_df, df_autogluon_tabular)

In [464]:

# import deep copy
from copy import deepcopy
class EnsambleTrainer:
    def __init__(self, models, previous_state=None):
        self.models = models
        self.model_weights = None
        self.train_results = None
        self.previous_state = previous_state # a result_df from a previous run, to avoid recomputing everything

    def _combina_results(self, results):
        from collections import defaultdict
        # Diccionario para almacenar resultados intermedios
        combined_results = defaultdict(dict)

        for split, models in results.items():
            for model_info in models:
                model_name = model_info["model"].name
                pred_df = model_info["pred_df"]

                for _, row in pred_df.iterrows():
                    key = (row["product_id"], row["date_id"])
                    combined_results[key]["product_id"] = row["product_id"]
                    combined_results[key]["date_id"] = row["date_id"]
                    combined_results[key]["target"] = row["target"]
                    combined_results[key][f"prediction_{model_name}"] = row["prediction"]

        # Convertir a DataFrame
        results_df = pd.DataFrame(combined_results.values())

        # Opcional: ordenar columnas
        cols = ["product_id", "date_id", "target"] + sorted([col for col in results_df.columns if col not in {"product_id", "date_id", "target"}])
        results_df = results_df[cols]

        def fill_row_na_with_row_mean(row, prediction_cols):
            preds = row[prediction_cols]
            #row[prediction_cols] = preds.fillna(preds.mean(skipna=True))

            #row[prediction_cols] = preds.fillna(-1000)  # Rellenar con -1000 para indicar que no hay predicción
            # rellena nans con la columna prediction_SMA-12 que es el fallback
            row[prediction_cols] = preds.fillna(row["prediction_SMA-12"])
            
            return row

        prediction_cols = [col for col in results_df.columns if col.startswith("prediction_")]

        results_df = results_df.apply(fill_row_na_with_row_mean, axis=1, prediction_cols=prediction_cols)   
        self.train_results = results_df
        return results_df

    def descombina_results(self, results_df, models, splitter, df_original):
        """
        Reconstruye el diccionario `results` tal como lo genera `fit`,
        usando el results_df de `_combina_results`, el splitter y los modelos.
    
        Parameters:
            results_df (pd.DataFrame): Output de `_combina_results`.
            models (list): Lista de modelos usados (con .name).
            splitter: El splitter con .split(df_original) disponible.
            df_original (pd.DataFrame): El dataframe original que se usó para el split.
    
        Returns:
            dict: Diccionario como {split_0: [...], split_1: [...], ...}
        """
    
        # Crear estructura vacía para los splits
        number_of_splits = splitter.get_n_splits(df_original)
        results = {f"split_{i}": [] for i in range(number_of_splits)}
    
        # Obtener los test date_ids de cada split
        split_test_dates = []
        for train_idx, test_idx in splitter.split(df_original):
            test_date_ids = df_original.iloc[test_idx]["date_id"].unique()
            if len(test_date_ids) != 1:
                raise ValueError("Cada split debe tener exactamente un único date_id en test.")
            split_test_dates.append(test_date_ids[0])
    
        # Para cada modelo, reconstruir sus predicciones por split
        for model in models:
            pred_col = f"prediction_{model.name}"
            if pred_col not in results_df.columns:
                continue  # el modelo no fue usado
            
            # Extraer filas para este modelo
            model_df = results_df[["product_id", "date_id", "target", pred_col]].copy()
            model_df = model_df.rename(columns={pred_col: "prediction"})
    
            # Asignar cada fila al split correspondiente según date_id
            for i, test_date_id in enumerate(split_test_dates):
                pred_df_split = model_df[model_df["date_id"] == test_date_id].copy()
                if pred_df_split.empty:
                    continue
                
                target_df = pred_df_split[["product_id", "date_id", "target"]].copy()
    
                results[f"split_{i}"].append({
                    "model": model,
                    "target": target_df,
                    "pred_df": pred_df_split
                })
    
        return results

    def _optimize_weights_mejorado(self, results_df, alpha=10.0):
        import numpy as np
        import pandas as pd
        from sklearn.linear_model import Ridge

        prediction_cols = [col for col in results_df.columns if col.startswith("prediction_")]

        weights_list = []
        predictions_list = []
        product_ids = results_df["product_id"].unique()

        for product_id in product_ids:
            product_rows = results_df[results_df["product_id"] == product_id]
            
            if len(product_rows) < 2:
                # No es suficiente para ajustar pesos, fallback a promedio simple
                avg_preds = product_rows[prediction_cols].mean().values
                weights = avg_preds / avg_preds.sum()
            else:
                X = product_rows[prediction_cols].values
                y = product_rows["target"].values

                # Ridge con regularización fuerte y sin intercepto
                model = Ridge(alpha=alpha, fit_intercept=False, positive=True)
                model.fit(X, y)
                weights = model.coef_
                if weights.sum() == 0:
                    weights = np.ones_like(weights) / len(weights)
                else:
                    weights = weights / weights.sum()

            # Guardar pesos como dict
            weights_dict = dict(zip(prediction_cols, weights))
            weights_list.append(weights_dict)

            # Hacer predicción promedio ponderada sobre las filas de esta serie
            weighted_preds = product_rows[prediction_cols].values @ weights
            mean_prediction = np.mean(weighted_preds)
            predictions_list.append(mean_prediction)

        agg_df = pd.DataFrame({
            "product_id": product_ids,
            "weights": weights_list,
            "predictions": predictions_list
        })

        if not hasattr(self, "model_weights") or self.model_weights is None:
            self.model_weights = {}

        # Podés usar un nombre especial para esta versión
        self.model_weights["ridge"] = agg_df[["product_id", "weights"]].set_index("product_id")


    def _optimize_weights(self, results_df, max_models=1):
        import numpy as np
        import pandas as pd
    
        prediction_cols = [col for col in results_df.columns if col.startswith("prediction_")]
    
        model_weights_by_max = {}
    
        product_weights = {}
        predictions_list = []
    
        for _, row in results_df.iterrows():
            product_id = row["product_id"]
            target = row["target"]
    
            preds = np.array([row[col] for col in prediction_cols])
            diffs = preds - target
    
            if max_models == 1:
                # comportamiento original: one-hot del modelo más cercano
                errors = diffs ** 2
                best_idx = np.argmin(errors)
                weights = np.zeros(len(prediction_cols))
                weights[best_idx] = 1
            else:
                above = np.where(diffs >= 0)[0]
                below = np.where(diffs < 0)[0]
    
                if len(above) > 0 and len(below) > 0:
                    i_above = above[np.argmin(diffs[above])]
                    i_below = below[np.argmax(diffs[below])]
                    p1, p2 = preds[i_below], preds[i_above]
    
                    # resolver w en target = w*p1 + (1-w)*p2 => w = (target - p2)/(p1 - p2)
                    denom = p1 - p2
                    if denom != 0:
                        w = (target - p2) / denom
                        w = np.clip(w, 0, 1)
                    else:
                        w = 0.5  # predicciones iguales => promedio
    
                    weights = np.zeros(len(prediction_cols))
                    weights[i_below] = w
                    weights[i_above] = 1 - w
                else:
                    # todos arriba o todos abajo: fallback a comportamiento original
                    errors = diffs ** 2
                    best_idx = np.argmin(errors)
                    weights = np.zeros(len(prediction_cols))
                    weights[best_idx] = 1
    
            if product_id not in product_weights:
                product_weights[product_id] = []
            product_weights[product_id].append(weights)
    
        weights_list = []
        predictions_list = []
        product_ids = list(product_weights.keys())
    
        for product_id in product_ids:
            weight_arr = np.mean(product_weights[product_id], axis=0)
            weights_dict = dict(zip(prediction_cols, weight_arr))
            weights_list.append(weights_dict)
    
            product_rows = results_df[results_df["product_id"] == product_id]
            preds_matrix = product_rows[prediction_cols].values
            weighted_preds = preds_matrix @ weight_arr
            mean_prediction = np.mean(weighted_preds)
            predictions_list.append(mean_prediction)
    
        agg_df = pd.DataFrame({
            "product_id": product_ids,
            "weights": weights_list,
            "predictions": predictions_list
        })
    
        if not hasattr(self, "model_weights") or self.model_weights is None:
            self.model_weights = {}
    
        self.model_weights[max_models] = agg_df[["product_id", "weights"]].set_index("product_id")


    def _compute_metrics_simple(self, y_true, y_pred):
        """Calcula el error absoluto medio entre y_true e y_pred"""
        return np.sum(np.abs(y_true - y_pred)) / np.sum(y_true) if np.sum(y_true) > 0 else 0

    def _compute_metrics(self, results_df, max_models=1):
        prediction_cols = [col for col in results_df.columns if col.startswith("prediction_")]
        agg_df = results_df.groupby("product_id")[["target"] + prediction_cols].sum().reset_index()
        agg_df = agg_df.set_index("product_id")
        agg_df["weights"] = self.model_weights[max_models]["weights"]
        agg_df["prediction_ensamble"] = agg_df.apply(
            lambda row: sum(row[col] * row["weights"][col] for col in prediction_cols), 
            axis=1
        )
        self.agg_df = agg_df
        # calculo metricas
        metrics = {}
        def total_error(y_true, y_pred):
            return np.sum(np.abs(y_true - y_pred)) / np.sum(y_true)

        prediction_cols = [col for col in agg_df.columns if col.startswith("prediction_")]
        for col in prediction_cols:
            metrics[col] = total_error(agg_df["target"], agg_df[col])
        return pd.DataFrame(metrics, index=[0]).T.rename(columns={0: "error"}).sort_values(by="error", ascending=True)
    
    def fit(self, df, splitter):
        """Entrena todos los modelos en cada split del splitter"""
        df = df.dropna(subset=["target"])
        number_of_splits = splitter.get_n_splits(df)
        results = {f"split_{i}": [] for i in range(number_of_splits)}
        if self.previous_state is not None:
            cached_results = self.descombina_results(
                self.previous_state,
                self.models,
                splitter,
                df
            )
        else:
            cached_results = {}

        for i, (train_idx, test_idx) in enumerate(splitter.split(df)):
            # Obtener las fechas de los splits originales
            train_dates = df.iloc[train_idx]["date_id"].unique()
            test_dates = df.iloc[test_idx]["date_id"].unique()
            
            for m in self.models:
                model = deepcopy(m)
                if f"split_{i}" in cached_results and model.name in [m["model"].name for m in cached_results[f"split_{i}"]]:
                    # Si el modelo ya fue entrenado en este split, lo uso del cache
                    cached_model = next(m for m in cached_results[f"split_{i}"] if m["model"].name == model.name)
                    results[f"split_{i}"].append(cached_model)
                    print(f"Modelo {model.name} ya entrenado en split {i+1}/{number_of_splits}, usando cache.")
                    continue
                # check if this split 
                df_model = model.prepare_dataset(df)
                
                # Recalcular train/test usando las fechas, no los índices
                train_df = df_model[df_model["date_id"].isin(train_dates)]
                test_df = df_model[df_model["date_id"].isin(test_dates)]
                test_df = test_df[test_df["product_id"].isin(product_ids)]
                
                pred_df = model.fit_and_predict(train_df, test_df, df_model)
                results[f"split_{i}"].append({
                    "model": model,
                    "target": test_df[["target", "product_id", "date_id"]],
                    "pred_df": pred_df
                })
                print(f"Modelo {model.name} entrenado en split {i+1}/{number_of_splits}:")
                print(self._compute_metrics_simple(pred_df["target"], pred_df["prediction"]))
        
        results_df = self._combina_results(results)
        # VALIDACION
        biggest_date = results_df["date_id"].max()
        val_weights_df = results_df[results_df["date_id"] != biggest_date]
        test_df = results_df[results_df["date_id"] == biggest_date]
        self._optimize_weights(val_weights_df)
        print("VALIDACIÓN 1 MODEL MAX (primer fold con pesos del resto):")
        print(self._compute_metrics(test_df))

        self._optimize_weights(val_weights_df, max_models=2)
        print("VALIDACIÓN 2 MODEL MAX (primer fold con pesos del resto):")
        print(self._compute_metrics(test_df, max_models=2).to_string())

        self._optimize_weights_mejorado(val_weights_df, alpha=5.0)
        print("VALIDACIÓN MEJORADO (primer fold con pesos del resto): ALPHA=5.0")
        print(self._compute_metrics(test_df, max_models="ridge").to_string())
        self._optimize_weights_mejorado(val_weights_df, alpha=10.0)
        print("VALIDACIÓN MEJORADO (primer fold con pesos del resto): ALPHA=10.0")
        print(self._compute_metrics(test_df, max_models="ridge").to_string())

        # OPTIMIZACION
        self._optimize_weights(results_df)
        #print(self._compute_metrics(results_df))
        self._optimize_weights(results_df, max_models=2)
        #print(self._compute_metrics(results_df, max_models=2))

        self._optimize_weights_mejorado(results_df)
        #print("VALIDACIÓN RIDGE (primer fold con pesos del resto):")
        #print(self._compute_metrics(test_df, max_models="ridge").to_string())
        #self._optimize_weights_mejorado(results_df, alpha=100.0)
        #print("VALIDACIÓN RIDGE (primer fold con pesos del resto, alpha=100):")
        #print(self._compute_metrics(test_df, max_models="ridge").to_string())
           

    def final_pred(self, df, kaggle_date_id, previous_final_df=None):
        """Vuelve a entrenar todos los modelos con el dataset completo"""
        results = {"split_final": []}
        for m in self.models:
            model = deepcopy(m)
            if previous_final_df is not None and f"prediction_{model.name}" in previous_final_df.columns:
                # Si el modelo ya fue entrenado en el final, lo uso del cache
                previous_final_df["date_id"] = kaggle_date_id
                pred_df = previous_final_df[["product_id", "date_id", "target", f"prediction_{model.name}"]].copy()
                pred_df = pred_df.rename(columns={f"prediction_{model.name}": "prediction"})
                results["split_final"].append({
                    "model": model,
                    "target": pred_df[["target", "product_id", "date_id"]],
                    "pred_df": pred_df
                })
                print(f"Modelo {model.name} ya entrenado en final, usando cache.")
                continue
            print(f"Entrenando modelo {model.name} para predicción final.")
            df_model = model.prepare_dataset(df)
            # Recalcular train/test usando las fechas, no los índices
            train_df = df_model[df_model["date_id"] < kaggle_date_id]
            train_df = train_df.dropna(subset=["target"])
            pred_df = df_model[df_model["date_id"] == kaggle_date_id]
            pred_df = pred_df[pred_df["product_id"].isin(product_ids)]

            pred_df = model.fit_and_predict(train_df, pred_df, df_model)
            results["split_final"].append({
                "model": model,
                "target": pred_df[["target", "product_id", "date_id"]],
                "pred_df": pred_df
            })
        results_df = self._combina_results(results)  
        prediction_cols = [col for col in results_df.columns if col.startswith("prediction_")]
        agg_df = results_df.groupby("product_id")[["target"] + prediction_cols].sum().reset_index()
        agg_df = agg_df.set_index("product_id", drop=False)
        agg_df["weights"] = self.model_weights[1]["weights"]
        agg_df["prediction_ensamble"] = agg_df.apply(
            lambda row: sum(row[col] * row["weights"][col] for col in prediction_cols), 
            axis=1
        )
        agg_df["weights"] = self.model_weights[2]["weights"]
        agg_df["prediction_ensamble_2"] = agg_df.apply(
            lambda row: sum(row[col] * row["weights"][col] for col in prediction_cols), 
            axis=1
        )
        agg_df["weights_ridge"] = self.model_weights["ridge"]["weights"]
        agg_df["prediction_ensamble_ridge"] = agg_df.apply(
            lambda row: sum(row[col] * row["weights_ridge"][col] for col in prediction_cols), 
            axis=1
        )
        return agg_df.rename(columns={"prediction_ensamble": "tn", "prediction_ensamble_2": "tn_2", "prediction_ensamble_ridge": "tn_ridge"})


In [465]:
previous_state = pd.read_csv("train_results_f1f3569368d057e634168ed327e8d607.csv")
previous_state

,product_id,date_id,target,prediction_AutoGluon-best_quality,prediction_LGBM-extra_trees-False-trials-1-scaling-False-boosting-dart-weight-False-target-t+2,prediction_LGBM-extra_trees-False-trials-1-scaling-False-boosting-dart-weight-True-target-t+2,prediction_LGBM-extra_trees-False-trials-1-scaling-False-boosting-gbdt-weight-False-target-t+2,prediction_LGBM-extra_trees-False-trials-1-scaling-False-boosting-gbdt-weight-True-target-logdiff,prediction_LGBM-extra_trees-False-trials-1-scaling-True-boosting-dart-weight-False-target-t+2,prediction_LGBM-extra_trees-False-trials-1-scaling-True-boosting-dart-weight-True-target-t+2,...,prediction_LGBM-extra_trees-True-trials-1-scaling-True-boosting-dart-weight-True-target-t+2,prediction_LGBM-extra_trees-True-trials-1-scaling-True-boosting-gbdt-weight-False-target-delta,prediction_LGBM-extra_trees-True-trials-1-scaling-True-boosting-gbdt-weight-False-target-t+2,prediction_LGBM-extra_trees-True-trials-1-scaling-True-boosting-gbdt-weight-True-target-delta,prediction_LGBM-extra_trees-True-trials-1-scaling-True-boosting-gbdt-weight-True-target-t+2,prediction_LinearRegression-magicos-['lags']-all,prediction_LinearRegressionCombination-magicos-['lags']-FOODS,prediction_LinearRegressionCombination-magicos-['lags']-HC,prediction_SMA-12,target_metamodel
0,20001.0,33.0,1504.688599,1512.528566,1248.137002,1265.288648,1262.658583,1391.704168,1556.323268,1416.853108,...,1385.755467,1566.726276,1469.350537,1413.007016,1413.243687,1195.374512,1195.374512,1127.511719,1487.869476,1313.990727
1,20002.0,33.0,1087.308594,1256.974084,1260.869115,1286.906964,1339.364460,1599.449699,1187.433564,1055.984514,...,1014.457444,1301.504709,1010.650623,1416.308874,1041.910153,1305.162598,1305.162598,1556.346436,1197.552073,1170.449434
2,20003.0,33.0,892.501282,986.016880,639.791459,623.227634,764.503507,711.995706,907.276354,802.678951,...,787.959410,881.143328,863.192569,722.764728,714.868486,668.048462,641.245361,668.048462,796.305669,840.955833
3,20004.0,33.0,637.900024,788.978117,518.467420,515.259466,604.995951,522.558861,768.962043,561.775104,...,580.162454,655.658182,695.042883,609.501255,509.368870,632.832825,509.441162,632.832825,629.387767,633.368590
4,20005.0,33.0,593.244446,726.746090,464.391654,445.572379,494.646220,556.171343,643.812018,505.359120,...,490.773926,573.483157,590.516448,592.244234,469.391819,738.727356,536.519348,738.727356,638.415240,587.851639
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3816,21263.0,29.0,0.033880,0.051955,0.020213,0.019122,0.142551,0.060767,0.016621,0.041974,...,0.062237,0.043699,0.033309,0.083656,0.114717,0.133198,0.133198,0.133198,0.133198,0.000000
3817,21265.0,29.0,0.015930,0.151885,0.074671,0.076610,0.187956,-0.044931,0.072726,0.095258,...,0.087998,0.071136,0.073858,0.043707,0.157576,0.151885,0.151885,0.151885,0.151885,0.000000
3818,21266.0,29.0,0.014800,0.151885,0.074870,0.076807,0.187956,-0.042092,0.073328,0.094781,...,0.089981,0.072081,0.073658,0.044069,0.153488,0.151885,0.151885,0.151885,0.151885,0.000000
3819,21267.0,29.0,0.040540,0.160505,0.076405,0.078244,0.183233,-0.023984,0.050088,0.078135,...,0.054450,0.087639,0.054250,0.078522,0.074571,0.160505,0.160505,0.160505,0.160505,0.000000


In [466]:
#drop todas las columnas que digan LinearRegression para que se vuelva a calcular el fallback
previous_state = previous_state.drop(columns=[col for col in previous_state.columns if "LinearRegression" in col], errors='ignore')

In [467]:
# dropeo columnas que dicen scaling-True porque las voy a recalcular
#previous_state = previous_state.drop(columns=[col for col in previous_state.columns if "scaling-True" in col], errors='ignore')

In [468]:
# TODO: entrenar el lightgbm con todos los product_ids? (la validacion solo con los 780)
import sys
from contextlib import redirect_stdout


# only linear regressions + moving average
trainer = EnsambleTrainer([

    #MODELOS QUE SI
    LinearRegressionModel(),
    LinearRegressionCombinationModel(especialidad="HC", product_ids="magicos"),
    LinearRegressionCombinationModel(especialidad="FOODS", product_ids="magicos"),
    BasicLGBMPredictor(n_trials=0, with_scaling=False, extra_trees=False, boosting_type="gbdt", use_weight=True, target="logdiff"),


    BasicLGBMPredictor(n_trials=0, with_scaling=True, extra_trees=True, boosting_type="dart", use_weight=False, target="t+2"),
    BasicLGBMPredictor(n_trials=0, with_scaling=True, extra_trees=True, boosting_type="gbdt", use_weight=False, target="t+2"),
    BasicLGBMPredictor(n_trials=0, with_scaling=True, extra_trees=False, boosting_type="dart", use_weight=True, target="t+2"),

    BasicLGBMPredictor(n_trials=0, with_scaling=True, extra_trees=True, boosting_type="gbdt", use_weight=False, target="delta"),
    BasicLGBMPredictor(n_trials=0, with_scaling=True, extra_trees=True, boosting_type="gbdt", use_weight=True, target="delta"),
    BasicLGBMPredictor(n_trials=0, with_scaling=True, extra_trees=True, boosting_type="dart", use_weight=True, target="delta"),

    BasicLGBMPredictor(n_trials=0, with_scaling=True, extra_trees=True, boosting_type="gbdt", use_weight=True, target="t+2"),
    BasicLGBMPredictor(n_trials=0, with_scaling=True, extra_trees=False, boosting_type="gbdt", use_weight=True, target="t+2"),
    BasicLGBMPredictor(n_trials=0, with_scaling=True, extra_trees=True, boosting_type="dart", use_weight=True, target="t+2"),
    BasicLGBMPredictor(n_trials=0, with_scaling=False, extra_trees=True, boosting_type="dart", use_weight=True, target="t+2"),
    BasicLGBMPredictor(n_trials=0, with_scaling=False, extra_trees=False, boosting_type="dart", use_weight=True, target="t+2"),
    BasicLGBMPredictor(n_trials=0, with_scaling=False, extra_trees=True, boosting_type="gbdt", use_weight=False, target="t+2"),
    BasicLGBMPredictor(n_trials=0, with_scaling=False, extra_trees=False, boosting_type="gbdt", use_weight=False, target="t+2"),
    BasicLGBMPredictor(n_trials=0, with_scaling=False, extra_trees=True, boosting_type="dart", use_weight=False, target="t+2"),
    BasicLGBMPredictor(n_trials=0, with_scaling=True, extra_trees=False, boosting_type="dart", use_weight=False, target="t+2"),
    BasicLGBMPredictor(n_trials=0, with_scaling=False, extra_trees=False, boosting_type="dart", use_weight=False, target="t+2"),
    
    AutoGluonPredictor(presets="best_quality"),
    SimpleMovingAveragePredictor(window_size=12),


], previous_state=previous_state)
splitter = CustomTimeSeriesSplit(n_splits=5, gap=1)
trainer.fit(df, splitter)


Registros de entrenamiento: 33
Modelo LinearRegression-magicos-['lags']-all entrenado en split 1/5:
0.33242905
product ids especialidad: [20001, 20002, 20006, 20008, 20010, 20011, 20017, 20018, 20021, 20026, 20028, 20035, 20039, 20042, 20045, 20049, 20051, 20053, 20055, 20180]
Registros de entrenamiento: 33
Modelo LinearRegressionCombination-magicos-['lags']-HC entrenado en split 1/5:
0.44432706
product ids especialidad: [20003, 20019, 20046, 20052, 20086]
Registros de entrenamiento: 33
Modelo LinearRegressionCombination-magicos-['lags']-FOODS entrenado en split 1/5:
0.3397572
Modelo LGBM-extra_trees-False-trials-1-scaling-False-boosting-gbdt-weight-True-target-logdiff ya entrenado en split 1/5, usando cache.
Modelo LGBM-extra_trees-True-trials-1-scaling-True-boosting-dart-weight-False-target-t+2 ya entrenado en split 1/5, usando cache.
Modelo LGBM-extra_trees-True-trials-1-scaling-True-boosting-gbdt-weight-False-target-t+2 ya entrenado en split 1/5, usando cache.
Modelo LGBM-extra_tre

In [469]:
# Expandir los diccionarios de weights en columnas separadas
weights_df = pd.json_normalize(trainer.model_weights["ridge"]['weights'].tolist())

# Agregar el product_id como índice
weights_df.index = trainer.model_weights["ridge"].index

# Resetear el índice si quieres product_id como columna
weights_df = weights_df.reset_index()

weights_df

,product_id,prediction_AutoGluon-best_quality,prediction_LGBM-extra_trees-False-trials-1-scaling-False-boosting-dart-weight-False-target-t+2,prediction_LGBM-extra_trees-False-trials-1-scaling-False-boosting-dart-weight-True-target-t+2,prediction_LGBM-extra_trees-False-trials-1-scaling-False-boosting-gbdt-weight-False-target-t+2,prediction_LGBM-extra_trees-False-trials-1-scaling-False-boosting-gbdt-weight-True-target-logdiff,prediction_LGBM-extra_trees-False-trials-1-scaling-True-boosting-dart-weight-False-target-t+2,prediction_LGBM-extra_trees-False-trials-1-scaling-True-boosting-dart-weight-True-target-t+2,prediction_LGBM-extra_trees-False-trials-1-scaling-True-boosting-gbdt-weight-True-target-t+2,prediction_LGBM-extra_trees-True-trials-1-scaling-False-boosting-dart-weight-False-target-t+2,...,prediction_LGBM-extra_trees-True-trials-1-scaling-True-boosting-dart-weight-True-target-delta,prediction_LGBM-extra_trees-True-trials-1-scaling-True-boosting-dart-weight-True-target-t+2,prediction_LGBM-extra_trees-True-trials-1-scaling-True-boosting-gbdt-weight-False-target-delta,prediction_LGBM-extra_trees-True-trials-1-scaling-True-boosting-gbdt-weight-False-target-t+2,prediction_LGBM-extra_trees-True-trials-1-scaling-True-boosting-gbdt-weight-True-target-delta,prediction_LGBM-extra_trees-True-trials-1-scaling-True-boosting-gbdt-weight-True-target-t+2,prediction_LinearRegression-magicos-['lags']-all,prediction_LinearRegressionCombination-magicos-['lags']-FOODS,prediction_LinearRegressionCombination-magicos-['lags']-HC,prediction_SMA-12
0,20001.0,0.064953,0.003198,0.004420,0.053120,0.059025,0.054916,0.030334,0.017161,0.003151,...,0.044952,0.045101,0.073619,0.053889,0.038295,0.042404,0.118360,0.118360,0.042098,0.035522
1,20002.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.500000,0.500000,0.000000,0.000000
2,20003.0,0.104800,0.000000,0.000000,0.021187,0.107573,0.033803,0.002302,0.000000,0.000000,...,0.008965,0.000000,0.133333,0.123890,0.008587,0.038761,0.064260,0.129460,0.064260,0.052652
3,20004.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000
4,20005.0,0.000000,0.000000,0.000000,0.000000,0.172532,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.049943,0.000000,0.777525
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
775,21252.0,0.073562,0.032782,0.034241,0.063278,0.037774,0.028892,0.029858,0.024649,0.048532,...,0.043775,0.023101,0.031114,0.025066,0.033991,0.022775,0.073562,0.073562,0.073562,0.073562
776,21265.0,0.049142,0.027454,0.030065,0.152076,0.029016,0.020814,0.028162,0.023061,0.049491,...,0.042247,0.020662,0.035837,0.016962,0.038685,0.019824,0.049142,0.049142,0.049142,0.049142
777,21266.0,0.050418,0.026787,0.029216,0.146447,0.021825,0.020093,0.028710,0.024413,0.048696,...,0.042706,0.021385,0.047206,0.017084,0.036889,0.019810,0.050418,0.050418,0.050418,0.050418
778,21267.0,0.057059,0.028800,0.031281,0.155353,0.021069,0.019835,0.023376,0.017118,0.052379,...,0.033143,0.016282,0.037380,0.014861,0.031538,0.014969,0.057059,0.057059,0.057059,0.057059


In [470]:
# Crear informe de pesos totales por modelo
weight_summary = weights_df.drop(columns=['product_id']).sum().reset_index()
weight_summary.columns = ['modelo', 'peso_total']
weight_summary = weight_summary.sort_values('peso_total', ascending=False)

print("Informe de Pesos Totales por Modelo:")
print("=" * 50)
weight_summary

Informe de Pesos Totales por Modelo:


,modelo,peso_total
21,prediction_SMA-12,92.738007
10,prediction_LGBM-extra_trees-True-trials-1-scal...,60.238154
4,prediction_LGBM-extra_trees-False-trials-1-sca...,55.337525
3,prediction_LGBM-extra_trees-False-trials-1-sca...,54.194307
18,prediction_LinearRegression-magicos-['lags']-all,44.784447
19,prediction_LinearRegressionCombination-magicos...,43.792780
20,prediction_LinearRegressionCombination-magicos...,38.612565
0,prediction_AutoGluon-best_quality,37.237607
1,prediction_LGBM-extra_trees-False-trials-1-sca...,32.089251
2,prediction_LGBM-extra_trees-False-trials-1-sca...,30.892708


In [ ]:
# calcular un summary total de pesos por tonelada de cada modelo, para ello hay que multiplicar los pesos por la tonelada de cada serie
toneladas_df = df.groupby("product_id")["tn"].sum().reset_index()
toneladas_df = toneladas_df.rename(columns={"tn": "tonelada_total"})
weights_df_2 = weights_df.merge(toneladas_df, on="product_id", how="left")
weights_df_2 = weights_df_2.set_index("product_id")
weights_summary = weights_df_2.drop(columns=['tonelada_total']).multiply(weights_df_2['tonelada_total'], axis=0).sum().reset_index()
weights_summary.columns = ['modelo', 'peso_por_tonelada']
weights_summary = weights_summary.sort_values('peso_por_tonelada', ascending=False)
print("Informe de Pesos por Tonelada por Modelo:")
print("=" * 50)
weights_summary

MergeError: Passing 'suffixes' which cause duplicate columns {'tonelada_total_x'} is not allowed.

In [471]:
weights_df.describe()

,product_id,prediction_AutoGluon-best_quality,prediction_LGBM-extra_trees-False-trials-1-scaling-False-boosting-dart-weight-False-target-t+2,prediction_LGBM-extra_trees-False-trials-1-scaling-False-boosting-dart-weight-True-target-t+2,prediction_LGBM-extra_trees-False-trials-1-scaling-False-boosting-gbdt-weight-False-target-t+2,prediction_LGBM-extra_trees-False-trials-1-scaling-False-boosting-gbdt-weight-True-target-logdiff,prediction_LGBM-extra_trees-False-trials-1-scaling-True-boosting-dart-weight-False-target-t+2,prediction_LGBM-extra_trees-False-trials-1-scaling-True-boosting-dart-weight-True-target-t+2,prediction_LGBM-extra_trees-False-trials-1-scaling-True-boosting-gbdt-weight-True-target-t+2,prediction_LGBM-extra_trees-True-trials-1-scaling-False-boosting-dart-weight-False-target-t+2,...,prediction_LGBM-extra_trees-True-trials-1-scaling-True-boosting-dart-weight-True-target-delta,prediction_LGBM-extra_trees-True-trials-1-scaling-True-boosting-dart-weight-True-target-t+2,prediction_LGBM-extra_trees-True-trials-1-scaling-True-boosting-gbdt-weight-False-target-delta,prediction_LGBM-extra_trees-True-trials-1-scaling-True-boosting-gbdt-weight-False-target-t+2,prediction_LGBM-extra_trees-True-trials-1-scaling-True-boosting-gbdt-weight-True-target-delta,prediction_LGBM-extra_trees-True-trials-1-scaling-True-boosting-gbdt-weight-True-target-t+2,prediction_LinearRegression-magicos-['lags']-all,prediction_LinearRegressionCombination-magicos-['lags']-FOODS,prediction_LinearRegressionCombination-magicos-['lags']-HC,prediction_SMA-12
count,780.000000,780.000000,780.000000,780.000000,780.000000,780.000000,780.000000,780.000000,780.000000,780.000000,...,780.000000,780.000000,780.000000,780.000000,780.000000,780.000000,780.000000,780.000000,780.000000,780.000000
mean,20541.421795,0.047741,0.041140,0.039606,0.069480,0.070946,0.028973,0.029724,0.034886,0.027361,...,0.029141,0.029510,0.035345,0.035720,0.032136,0.034593,0.057416,0.056145,0.049503,0.118895
std,353.984342,0.071952,0.071620,0.061760,0.079217,0.131594,0.044611,0.053596,0.072861,0.034837,...,0.060593,0.049951,0.059050,0.068543,0.052778,0.069204,0.081709,0.087440,0.072717,0.160633
min,20001.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,20238.750000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.035092
50%,20511.500000,0.020260,0.022985,0.025274,0.060705,0.036167,0.016264,0.011954,0.001397,0.015433,...,0.007521,0.014624,0.006814,0.020414,0.012995,0.012445,0.022227,0.020416,0.017121,0.066352
75%,20818.500000,0.057074,0.053290,0.054107,0.103783,0.063609,0.044153,0.041877,0.042214,0.047529,...,0.045917,0.044908,0.050615,0.047170,0.048086,0.042608,0.080987,0.076886,0.068021,0.157731
max,21276.000000,0.537406,1.000000,0.551808,0.677108,1.000000,0.544669,0.650754,0.700001,0.288029,...,0.954517,0.911557,0.476474,1.000000,0.723791,0.979727,0.500000,1.000000,0.544281,1.000000


In [472]:
trainer._compute_metrics(trainer.train_results)

,error
prediction_ensamble,0.107241
prediction_AutoGluon-best_quality,0.130428
prediction_LGBM-extra_trees-True-trials-1-scaling-True-boosting-dart-weight-True-target-delta,0.133041
prediction_LGBM-extra_trees-True-trials-1-scaling-True-boosting-gbdt-weight-True-target-delta,0.139982
prediction_LGBM-extra_trees-True-trials-1-scaling-False-boosting-gbdt-weight-False-target-t+2,0.140838
prediction_LGBM-extra_trees-True-trials-1-scaling-True-boosting-dart-weight-False-target-t+2,0.146662
prediction_LGBM-extra_trees-False-trials-1-scaling-True-boosting-dart-weight-False-target-t+2,0.147237
prediction_LGBM-extra_trees-True-trials-1-scaling-True-boosting-gbdt-weight-False-target-t+2,0.147330
prediction_LGBM-extra_trees-False-trials-1-scaling-False-boosting-gbdt-weight-False-target-t+2,0.150029
prediction_LGBM-extra_trees-True-trials-1-scaling-True-boosting-gbdt-weight-False-target-delta,0.152136


In [473]:
trainer.train_results.columns

Index(['product_id', 'date_id', 'target', 'prediction_AutoGluon-best_quality',
       'prediction_LGBM-extra_trees-False-trials-1-scaling-False-boosting-dart-weight-False-target-t+2',
       'prediction_LGBM-extra_trees-False-trials-1-scaling-False-boosting-dart-weight-True-target-t+2',
       'prediction_LGBM-extra_trees-False-trials-1-scaling-False-boosting-gbdt-weight-False-target-t+2',
       'prediction_LGBM-extra_trees-False-trials-1-scaling-False-boosting-gbdt-weight-True-target-logdiff',
       'prediction_LGBM-extra_trees-False-trials-1-scaling-True-boosting-dart-weight-False-target-t+2',
       'prediction_LGBM-extra_trees-False-trials-1-scaling-True-boosting-dart-weight-True-target-t+2',
       'prediction_LGBM-extra_trees-False-trials-1-scaling-True-boosting-gbdt-weight-True-target-t+2',
       'prediction_LGBM-extra_trees-True-trials-1-scaling-False-boosting-dart-weight-False-target-t+2',
       'prediction_LGBM-extra_trees-True-trials-1-scaling-False-boosting-dart-weight-

In [474]:
def entrenar_metamodelo_stacking(results_df, model_type="lgbm", alpha=10.0):
    from sklearn.linear_model import Ridge
    from lightgbm import LGBMRegressor
    import pandas as pd
    from sklearn.linear_model import ElasticNet



    prediction_cols = [col for col in results_df.columns if col.startswith("prediction_")] + ["product_id"]
    X = results_df[prediction_cols].values
    y = results_df["target"].values

    if model_type == "ridge":
        meta_model = Ridge(alpha=alpha, fit_intercept=True)
    elif model_type == "lgbm":
        # import standarscaler
        meta_model = LGBMRegressor(n_estimators=500, learning_rate=0.05, num_leaves=8, linear_tree=True,)
    elif model_type == "elasticnet":
        meta_model = ElasticNet(alpha=alpha, l1_ratio=0.5, fit_intercept=True)
    elif model_type == "linear":
        from sklearn.linear_model import LinearRegression
        meta_model = LinearRegression(fit_intercept=True)
    elif model_type == "MLP":
        from sklearn.neural_network import MLPRegressor
        meta_model = MLPRegressor(hidden_layer_sizes=(100,), max_iter=500, random_state=42)
    else:
        raise ValueError(f"Modelo {model_type} no soportado")

    
    meta_model.fit(X, y)

    # Guardamos el modelo entrenado
    meta_model = meta_model
    meta_model_features = prediction_cols  # para mantener compatibilidad con predicciones
    return meta_model, meta_model_features

meta_model, meta_model_features = entrenar_metamodelo_stacking(trainer.train_results, model_type="lgbm")
print("Meta-modelo entrenado con éxito.")


Meta-modelo entrenado con éxito.


In [475]:
trainer.train_results["target_metamodel"] = meta_model.predict(trainer.train_results[meta_model_features].values).clip(min=0)


/home/fede/programacion/labo3/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [476]:
trainer.train_results[["target", "target_metamodel", "prediction_SMA-12"]]

,target,target_metamodel,prediction_SMA-12
0,1504.688599,1313.990727,1487.869476
1,1087.308594,1170.449434,1197.552073
2,892.501282,840.955833,796.305669
3,637.900024,633.368590,629.387767
4,593.244446,587.851639,638.415240
...,...,...,...
3816,0.033880,0.000000,0.133198
3817,0.015930,0.000000,0.151885
3818,0.014800,0.000000,0.151885
3819,0.040540,0.000000,0.160505


In [477]:

models_used = [model.name for model in trainer.models]
models_used = " ".join(models_used)
# hago un hash en base de models_used para el nombre del archivo
import hashlib
hash_object = hashlib.md5(models_used.encode())
hash_hex = hash_object.hexdigest()

trainer.train_results.to_csv(f"train_results_{hash_hex}.csv", index=False)

In [478]:
hash_hex

'f1f3569368d057e634168ed327e8d607'

In [479]:
trainer.agg_df.columns

Index(['target', 'prediction_AutoGluon-best_quality',
       'prediction_LGBM-extra_trees-False-trials-1-scaling-False-boosting-dart-weight-False-target-t+2',
       'prediction_LGBM-extra_trees-False-trials-1-scaling-False-boosting-dart-weight-True-target-t+2',
       'prediction_LGBM-extra_trees-False-trials-1-scaling-False-boosting-gbdt-weight-False-target-t+2',
       'prediction_LGBM-extra_trees-False-trials-1-scaling-False-boosting-gbdt-weight-True-target-logdiff',
       'prediction_LGBM-extra_trees-False-trials-1-scaling-True-boosting-dart-weight-False-target-t+2',
       'prediction_LGBM-extra_trees-False-trials-1-scaling-True-boosting-dart-weight-True-target-t+2',
       'prediction_LGBM-extra_trees-False-trials-1-scaling-True-boosting-gbdt-weight-True-target-t+2',
       'prediction_LGBM-extra_trees-True-trials-1-scaling-False-boosting-dart-weight-False-target-t+2',
       'prediction_LGBM-extra_trees-True-trials-1-scaling-False-boosting-dart-weight-True-target-t+2',
       

In [480]:
previous_final_df = pd.read_csv("final_df_f1f3569368d057e634168ed327e8d607.csv")
previous_final_df

# drop todas las columnas que digan LinearRegression para que se vuelva a calcular el fallback
previous_final_df = previous_final_df.drop(columns=[col for col in previous_final_df.columns if "LinearRegression" in col], errors='ignore')


In [481]:
#previous_final_df = previous_final_df.drop(columns=[col for col in previous_final_df.columns if "scaling-True" in col], errors='ignore')

In [482]:
final_df = trainer.final_pred(df, 35, previous_final_df=previous_final_df)

Entrenando modelo LinearRegression-magicos-['lags']-all para predicción final.
Registros de entrenamiento: 33
Entrenando modelo LinearRegressionCombination-magicos-['lags']-HC para predicción final.
product ids especialidad: [20001, 20002, 20006, 20008, 20010, 20011, 20017, 20018, 20021, 20026, 20028, 20035, 20039, 20042, 20045, 20049, 20051, 20053, 20055, 20180]
Registros de entrenamiento: 33
Entrenando modelo LinearRegressionCombination-magicos-['lags']-FOODS para predicción final.
product ids especialidad: [20003, 20019, 20046, 20052, 20086]
Registros de entrenamiento: 33
Modelo LGBM-extra_trees-False-trials-1-scaling-False-boosting-gbdt-weight-True-target-logdiff ya entrenado en final, usando cache.
Modelo LGBM-extra_trees-True-trials-1-scaling-True-boosting-dart-weight-False-target-t+2 ya entrenado en final, usando cache.
Modelo LGBM-extra_trees-True-trials-1-scaling-True-boosting-gbdt-weight-False-target-t+2 ya entrenado en final, usando cache.
Modelo LGBM-extra_trees-False-trial

In [483]:
final_df

,product_id,target,prediction_AutoGluon-best_quality,prediction_LGBM-extra_trees-False-trials-1-scaling-False-boosting-dart-weight-False-target-t+2,prediction_LGBM-extra_trees-False-trials-1-scaling-False-boosting-dart-weight-True-target-t+2,prediction_LGBM-extra_trees-False-trials-1-scaling-False-boosting-gbdt-weight-False-target-t+2,prediction_LGBM-extra_trees-False-trials-1-scaling-False-boosting-gbdt-weight-True-target-logdiff,prediction_LGBM-extra_trees-False-trials-1-scaling-True-boosting-dart-weight-False-target-t+2,prediction_LGBM-extra_trees-False-trials-1-scaling-True-boosting-dart-weight-True-target-t+2,prediction_LGBM-extra_trees-False-trials-1-scaling-True-boosting-gbdt-weight-True-target-t+2,...,prediction_LGBM-extra_trees-True-trials-1-scaling-True-boosting-gbdt-weight-True-target-t+2,prediction_LinearRegression-magicos-['lags']-all,prediction_LinearRegressionCombination-magicos-['lags']-FOODS,prediction_LinearRegressionCombination-magicos-['lags']-HC,prediction_SMA-12,weights,tn,tn_2,weights_ridge,tn_ridge
product_id,,,,,,,,,,,,,,,,,,,,,
20001.0,20001.0,0.0,1317.507170,1181.747647,1181.747647,1367.074970,1343.879339,1314.346070,1309.226625,1380.399528,...,1302.790620,1162.707520,1162.707520,1199.431152,1454.732737,{'prediction_AutoGluon-best_quality': 0.136413...,1338.639695,1338.493736,{'prediction_AutoGluon-best_quality': 0.064952...,1303.024175
20002.0,20002.0,0.0,1102.049512,1167.432186,1167.432187,1346.510526,1419.823300,1142.880621,1137.664085,1027.065295,...,1270.694715,1183.640625,1183.640625,1294.223633,1175.437134,"{'prediction_AutoGluon-best_quality': 0.0, 'pr...",1237.864047,1260.220459,"{'prediction_AutoGluon-best_quality': 0.0, 'pr...",1183.640625
20003.0,20003.0,0.0,692.453958,658.460727,658.460728,632.810920,605.797608,761.680216,931.650708,728.497584,...,796.850950,684.763855,643.368164,684.763855,784.976405,{'prediction_AutoGluon-best_quality': 0.130158...,688.375431,681.680580,{'prediction_AutoGluon-best_quality': 0.104800...,736.665856
20004.0,20004.0,0.0,522.661770,557.094129,557.094131,544.855566,608.480333,548.375991,616.951158,595.137214,...,604.727901,580.485046,501.384521,580.485046,627.215322,"{'prediction_AutoGluon-best_quality': 0.2, 'pr...",545.993190,545.816522,"{'prediction_AutoGluon-best_quality': 0.0, 'pr...",501.384521
20005.0,20005.0,0.0,488.323007,547.798850,547.798851,548.287994,605.548100,483.163622,534.730229,561.524949,...,583.984579,563.560852,441.741516,563.560852,668.270111,"{'prediction_AutoGluon-best_quality': 0.2, 'pr...",550.268768,551.723243,"{'prediction_AutoGluon-best_quality': 0.0, 'pr...",646.135061
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21263.0,21263.0,0.0,-0.000404,0.018691,0.018691,0.016714,-0.011016,0.011367,0.020686,0.010275,...,0.011472,0.467764,0.467764,0.467764,0.029993,"{'prediction_AutoGluon-best_quality': 0.0, 'pr...",0.014973,0.014873,{'prediction_AutoGluon-best_quality': 0.035414...,0.095622
21265.0,21265.0,0.0,0.089541,0.043563,0.043563,0.044660,0.050531,0.034830,0.057535,0.063207,...,0.050923,0.089541,0.089541,0.089541,0.089541,{'prediction_AutoGluon-best_quality': 0.192185...,0.075863,0.073263,{'prediction_AutoGluon-best_quality': 0.049142...,0.072015
21266.0,21266.0,0.0,0.094659,0.045977,0.045977,0.044603,0.053446,0.034972,0.061862,0.057612,...,0.057835,0.094659,0.094659,0.094659,0.094659,{'prediction_AutoGluon-best_quality': 0.170837...,0.083523,0.079252,{'prediction_AutoGluon-best_quality': 0.050418...,0.075956


In [484]:
final_df.to_csv(f"final_df_{hash_hex}.csv", index=False)

In [485]:
meta_model.predict(final_df[meta_model_features])

array([ 1.25746589e+03,  1.26581489e+03,  8.35688784e+02,  9.10070829e+02,
        8.69959335e+02,  3.02701765e+02,  3.64347697e+02,  4.04772279e+02,
        6.24972297e+02,  3.42471222e+02,  3.31681035e+02,  2.43158771e+02,
        3.19547601e+02,  3.96110343e+02,  3.47913203e+02,  2.65316723e+02,
        2.40550879e+02,  2.94793656e+02,  3.13491610e+02,  3.36389501e+02,
        2.23224687e+02,  2.40309709e+02,  1.84248928e+02,  1.69230812e+02,
        1.89788213e+02,  2.26162121e+02,  1.59436638e+02,  1.45274094e+02,
        1.83800715e+02,  1.07147835e+02,  1.61318689e+02,  5.04670929e+02,
        1.60717169e+02,  1.29583949e+02,  7.41235326e+01,  1.26882793e+02,
        1.52205821e+02,  7.41365156e+01,  1.95099274e+02,  1.29709472e+02,
        1.35153062e+02,  1.54386191e+02,  1.23114034e+02,  1.71530983e+02,
        1.89945294e+02,  1.10702200e+02,  1.35863390e+02,  1.01143759e+02,
        1.65400804e+02,  1.16779262e+02,  9.02661957e+01,  7.95018017e+01,
        1.18814491e+02,  

In [486]:
final_df["prediccion_metamodelo"] = meta_model.predict(final_df[meta_model_features].values).clip(min=0)
final_df["prediccion_metamodelo"]

/home/fede/programacion/labo3/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


product_id
20001.0    1257.465890
20002.0    1265.814887
20003.0     835.688784
20004.0     910.070829
20005.0     869.959335
              ...     
21263.0       0.000000
21265.0       0.000000
21266.0       0.000000
21267.0       0.000000
21276.0       0.000000
Name: prediccion_metamodelo, Length: 780, dtype: float64

In [487]:

models_used = [model.name for model in trainer.models]
models_used = " ".join(models_used)
# hago un hash en base de models_used para el nombre del archivo
import hashlib
hash_object = hashlib.md5(models_used.encode())
hash_hex = hash_object.hexdigest()
description = f"Ensamble de modelos: {models_used}"
# save txt with name hash_hex.txt and the description
with open(f"description_{hash_hex}.txt", "w") as f:
    f.write(description)
submission = final_df[["product_id", "tn"]].reset_index(drop=True)
submission.to_csv(f"submission_weighted_ensamble_{hash_hex}.csv", index=False)
submission

,product_id,tn
0,20001.0,1338.639695
1,20002.0,1237.864047
2,20003.0,688.375431
3,20004.0,545.993190
4,20005.0,550.268768
...,...,...
775,21263.0,0.014973
776,21265.0,0.075863
777,21266.0,0.083523
778,21267.0,0.038094


In [488]:
submission_2_models = final_df[["product_id", "tn_2"]].reset_index(drop=True)
submission_2_models.rename(columns={"tn_2": "tn"}, inplace=True)
submission_2_models.to_csv(f"submission_2_models_weighted_ensamble_{hash_hex}.csv", index=False)
submission_2_models

,product_id,tn
0,20001.0,1338.493736
1,20002.0,1260.220459
2,20003.0,681.680580
3,20004.0,545.816522
4,20005.0,551.723243
...,...,...
775,21263.0,0.014873
776,21265.0,0.073263
777,21266.0,0.079252
778,21267.0,0.041415


In [489]:
submission_ridge = final_df[["product_id", "tn_ridge"]].reset_index(drop=True)
submission_ridge.rename(columns={"tn_ridge": "tn"}, inplace=True)
submission_ridge.to_csv(f"submission_ridge_weighted_ensamble_{hash_hex}.csv", index=False)
submission_ridge

,product_id,tn
0,20001.0,1303.024175
1,20002.0,1183.640625
2,20003.0,736.665856
3,20004.0,501.384521
4,20005.0,646.135061
...,...,...
775,21263.0,0.095622
776,21265.0,0.072015
777,21266.0,0.075956
778,21267.0,0.064979


In [490]:
submission_metamodel = final_df[["product_id", "prediccion_metamodelo"]].reset_index(drop=True)
submission_metamodel.rename(columns={"prediccion_metamodelo": "tn"}, inplace=True)
submission_metamodel.to_csv(f"submission_metamodel_weighted_ensamble_{hash_hex}.csv", index=False)
submission_metamodel

,product_id,tn
0,20001.0,1257.465890
1,20002.0,1265.814887
2,20003.0,835.688784
3,20004.0,910.070829
4,20005.0,869.959335
...,...,...
775,21263.0,0.000000
776,21265.0,0.000000
777,21266.0,0.000000
778,21267.0,0.000000


In [491]:
# hago un submit por cada modelo de final_df
for col in final_df.columns:
    if col.startswith("prediction_"):
        model_name = col.split("_", 1)[1]
        submission_model = final_df[["product_id", col]].rename(columns={col: "tn"}).reset_index(drop=True)
        submission_model.to_csv(f"submission_{model_name}_ensemble_{hash_hex}.csv", index=False)
        print(f"Submission for {model_name} saved as submission_{model_name}_ensemble_{hash_hex}.csv")

Submission for AutoGluon-best_quality saved as submission_AutoGluon-best_quality_ensemble_f1f3569368d057e634168ed327e8d607.csv
Submission for LGBM-extra_trees-False-trials-1-scaling-False-boosting-dart-weight-False-target-t+2 saved as submission_LGBM-extra_trees-False-trials-1-scaling-False-boosting-dart-weight-False-target-t+2_ensemble_f1f3569368d057e634168ed327e8d607.csv
Submission for LGBM-extra_trees-False-trials-1-scaling-False-boosting-dart-weight-True-target-t+2 saved as submission_LGBM-extra_trees-False-trials-1-scaling-False-boosting-dart-weight-True-target-t+2_ensemble_f1f3569368d057e634168ed327e8d607.csv
Submission for LGBM-extra_trees-False-trials-1-scaling-False-boosting-gbdt-weight-False-target-t+2 saved as submission_LGBM-extra_trees-False-trials-1-scaling-False-boosting-gbdt-weight-False-target-t+2_ensemble_f1f3569368d057e634168ed327e8d607.csv
Submission for LGBM-extra_trees-False-trials-1-scaling-False-boosting-gbdt-weight-True-target-logdiff saved as submission_LGBM-e

In [492]:
print(description)

Ensamble de modelos: LinearRegression-magicos-['lags']-all LinearRegressionCombination-magicos-['lags']-HC LinearRegressionCombination-magicos-['lags']-FOODS LGBM-extra_trees-False-trials-1-scaling-False-boosting-gbdt-weight-True-target-logdiff LGBM-extra_trees-True-trials-1-scaling-True-boosting-dart-weight-False-target-t+2 LGBM-extra_trees-True-trials-1-scaling-True-boosting-gbdt-weight-False-target-t+2 LGBM-extra_trees-False-trials-1-scaling-True-boosting-dart-weight-True-target-t+2 LGBM-extra_trees-True-trials-1-scaling-True-boosting-gbdt-weight-False-target-delta LGBM-extra_trees-True-trials-1-scaling-True-boosting-gbdt-weight-True-target-delta LGBM-extra_trees-True-trials-1-scaling-True-boosting-dart-weight-True-target-delta LGBM-extra_trees-True-trials-1-scaling-True-boosting-gbdt-weight-True-target-t+2 LGBM-extra_trees-False-trials-1-scaling-True-boosting-gbdt-weight-True-target-t+2 LGBM-extra_trees-True-trials-1-scaling-True-boosting-dart-weight-True-target-t+2 LGBM-extra_tree

In [493]:
agg_df = trainer.agg_df.copy()
pred_columns = [col for col in agg_df.columns if col.startswith("prediction_") if col != "prediction_ensamble"]

# entreno un linear regression para usar las predicciones y hacer una prediccion final
from sklearn.linear_model import LinearRegression
X = agg_df[pred_columns]
y = agg_df["target"]
model = LinearRegression()
model.fit(X, y)
agg_df["prediction_final"] = model.predict(X)
total_error = np.sum(np.abs(agg_df["target"] - agg_df["prediction_final"])) / np.sum(agg_df["target"])
print(f"Total error: {total_error}")

# calculo weights para cada product_id usando linear regression en lugar de scipy.minimize
from scipy.optimize import minimize

def optimize_weights_per_product(y_true, y_pred_values):
    """Optimiza los pesos para UNA SOLA fila usando scipy.minimize"""
    predictions = np.array(y_pred_values)
    
    if np.allclose(predictions, 0) or y_true == 0:
        # Si todas las predicciones son 0 o target es 0, usar pesos uniformes
        return np.ones(len(predictions)) / len(predictions)
    
    def objective(weights):
        """Función objetivo: error absoluto"""
        weighted_pred = np.dot(predictions, weights)
        return abs(y_true - weighted_pred)
    
    # Restricciones
    constraints = {'type': 'eq', 'fun': lambda w: np.sum(w) - 1}  # Suma = 1
    bounds = [(0, 1) for _ in range(len(predictions))]  # Pesos entre 0 y 1
    initial_weights = np.ones(len(predictions)) / len(predictions)  # Pesos iniciales uniformes
    
    try:
        result = minimize(objective, initial_weights, method='SLSQP', 
                         bounds=bounds, constraints=constraints)
        if result.success:
            return result.x
        else:
            return initial_weights
    except:
        return initial_weights

# Aplicar la optimización
agg_df["weights_2"] = agg_df.apply(
    lambda row: optimize_weights_per_product(row["target"], row[pred_columns].values), 
    axis=1
)

# Calcular la predicción ponderada
agg_df["prediction_final_2"] = agg_df.apply(
    lambda row: np.dot(row[pred_columns].values, row["weights_2"]), 
    axis=1
)

total_error_2 = np.sum(np.abs(agg_df["target"] - agg_df["prediction_final_2"])) / np.sum(agg_df["target"])
print(f"Total error with scipy optimized weights: {total_error_2}")
agg_df

final_df_3 = final_df.copy()
final_df_3["weights_2"] = agg_df["weights_2"]
final_df_3["pred_weights"] = final_df_3.apply(
    lambda row: np.dot(row[pred_columns].values, row["weights_2"]),
    axis=1
)
final_df_3["pred_weights"] = final_df_3["pred_weights"].clip(lower=0)  # Aseguro que la prediccion no sea negativa
final_df_3


Total error: 0.09204842835564173
Total error with scipy optimized weights: 0.031378568280190394


,product_id,target,prediction_AutoGluon-best_quality,prediction_LGBM-extra_trees-False-trials-1-scaling-False-boosting-dart-weight-False-target-t+2,prediction_LGBM-extra_trees-False-trials-1-scaling-False-boosting-dart-weight-True-target-t+2,prediction_LGBM-extra_trees-False-trials-1-scaling-False-boosting-gbdt-weight-False-target-t+2,prediction_LGBM-extra_trees-False-trials-1-scaling-False-boosting-gbdt-weight-True-target-logdiff,prediction_LGBM-extra_trees-False-trials-1-scaling-True-boosting-dart-weight-False-target-t+2,prediction_LGBM-extra_trees-False-trials-1-scaling-True-boosting-dart-weight-True-target-t+2,prediction_LGBM-extra_trees-False-trials-1-scaling-True-boosting-gbdt-weight-True-target-t+2,...,prediction_LinearRegressionCombination-magicos-['lags']-HC,prediction_SMA-12,weights,tn,tn_2,weights_ridge,tn_ridge,prediccion_metamodelo,weights_2,pred_weights
product_id,,,,,,,,,,,,,,,,,,,,,
20001.0,20001.0,0.0,1317.507170,1181.747647,1181.747647,1367.074970,1343.879339,1314.346070,1309.226625,1380.399528,...,1199.431152,1454.732737,{'prediction_AutoGluon-best_quality': 0.136413...,1338.639695,1338.493736,{'prediction_AutoGluon-best_quality': 0.064952...,1303.024175,1257.465890,"[0.02118768846420507, 0.021151327610012698, 0....",1226.105334
20002.0,20002.0,0.0,1102.049512,1167.432186,1167.432187,1346.510526,1419.823300,1142.880621,1137.664085,1027.065295,...,1294.223633,1175.437134,"{'prediction_AutoGluon-best_quality': 0.0, 'pr...",1237.864047,1260.220459,"{'prediction_AutoGluon-best_quality': 0.0, 'pr...",1183.640625,1265.814887,"[0.017870712700077274, 0.017897521977755678, 0...",1273.954148
20003.0,20003.0,0.0,692.453958,658.460727,658.460728,632.810920,605.797608,761.680216,931.650708,728.497584,...,684.763855,784.976405,{'prediction_AutoGluon-best_quality': 0.130158...,688.375431,681.680580,{'prediction_AutoGluon-best_quality': 0.104800...,736.665856,835.688784,"[0.028899460901653434, 0.028907541074980975, 0...",719.333954
20004.0,20004.0,0.0,522.661770,557.094129,557.094131,544.855566,608.480333,548.375991,616.951158,595.137214,...,580.485046,627.215322,"{'prediction_AutoGluon-best_quality': 0.2, 'pr...",545.993190,545.816522,"{'prediction_AutoGluon-best_quality': 0.0, 'pr...",501.384521,910.070829,"[7.920611427927976e-08, 4.89304132814765e-12, ...",548.376353
20005.0,20005.0,0.0,488.323007,547.798850,547.798851,548.287994,605.548100,483.163622,534.730229,561.524949,...,563.560852,668.270111,"{'prediction_AutoGluon-best_quality': 0.2, 'pr...",550.268768,551.723243,"{'prediction_AutoGluon-best_quality': 0.0, 'pr...",646.135061,869.959335,"[0.0, 1.1298455529916235e-08, 1.19653134137996...",563.560854
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21263.0,21263.0,0.0,-0.000404,0.018691,0.018691,0.016714,-0.011016,0.011367,0.020686,0.010275,...,0.467764,0.029993,"{'prediction_AutoGluon-best_quality': 0.0, 'pr...",0.014973,0.014873,{'prediction_AutoGluon-best_quality': 0.035414...,0.095622,0.000000,"[5.680301737136799e-17, 0.147468569205689, 0.1...",0.012468
21265.0,21265.0,0.0,0.089541,0.043563,0.043563,0.044660,0.050531,0.034830,0.057535,0.063207,...,0.089541,0.089541,{'prediction_AutoGluon-best_quality': 0.192185...,0.075863,0.073263,{'prediction_AutoGluon-best_quality': 0.049142...,0.072015,0.000000,"[2.7446918942111925e-09, 0.08130165334189599, ...",0.051849
21266.0,21266.0,0.0,0.094659,0.045977,0.045977,0.044603,0.053446,0.034972,0.061862,0.057612,...,0.094659,0.094659,{'prediction_AutoGluon-best_quality': 0.170837...,0.083523,0.079252,{'prediction_AutoGluon-best_quality': 0.050418...,0.075956,0.000000,"[0.00496907039025711, 0.09297367590062913, 0.0...",0.055357


In [494]:
submission_3 = final_df_3[["product_id", "pred_weights"]].rename(columns={"pred_weights": "tn"}).reset_index(drop=True)
submission_3.to_csv(f"submission_weights_ensamble_per_product_{hash_hex}.csv", index=False)
submission_3

,product_id,tn
0,20001.0,1226.105334
1,20002.0,1273.954148
2,20003.0,719.333954
3,20004.0,548.376353
4,20005.0,563.560854
...,...,...
775,21263.0,0.012468
776,21265.0,0.051849
777,21266.0,0.055357
778,21267.0,0.036825
